# Phân tích và chuẩn bị dữ liệu phát hiện ký tự — YOLO2

Notebook thực hành riêng cho nhiệm vụ **YOLO2** của đồ án nhận diện biển số (chạy trên Google Colab hoặc cục bộ).

- **Nguồn duy nhất:** `01_Du_lieu_goc/Roboflow/roboflow_ocr_v1_yolov8.zip` (`licence-plate-vn/ocr-stypd`, phiên bản 1; mốc kiểm kê cần đối chiếu: **20.262 ảnh** gồm `16.209` train, `2.027` val, `2.026` test). Không gộp Winter2897 hoặc Global v13.
- **Vai trò YOLO2 vs OCR:** Bộ xuất `yolo2_char_num` dùng **1 lớp duy nhất `0: char_num`** để phát hiện vị trí ký tự (không loại khung chỉ vì ký tự nằm ngoài 32 lớp OCR). Toàn bộ bảng lớp gốc và nhãn ký tự nguyên bản được **bảo toàn** trong gói `Ban_giao_OCR/` để bàn giao cho thành viên OCR.
- **Thư mục dự án trên Google Drive:** [Mở thư mục `Do_an_bien_so`](https://drive.google.com/drive/u/1/folders/1nBzyrK4yzCwul_6WSiItAKXqjl21DEw6). Nếu đây là thư mục được chia sẻ, hãy bấm **Thêm lối tắt vào Drive của tôi** (ví dụ vào `MyDrive/ML/Do_an_bien_so` hoặc `MyDrive/Do_an_bien_so`) để Colab đọc được qua `/content/drive/MyDrive/...`.
- **Không sửa tệp nguồn và không sửa notebook tổng/YOLO1.**

### Hai chế độ và thứ tự chạy theo mốc

| Chế độ / Mốc | Ý nghĩa |
|---|---|
| `CHE_DO = "xem_bao_cao"` *(mặc định khi bàn giao)* | Chỉ mở bảng, hình và kết luận đã lưu. Thiếu báo cáo sẽ thông báo rõ, **không tự quét ZIP hay giải mã ảnh**. |
| `CHE_DO = "xu_ly_du_lieu"` | Chạy kiểm tra, lưu tiến độ từng phần (`.jsonl.gz`), hỗ trợ chạy tiếp sau ngắt và cache đúng phiên bản. |
| **Mốc A** | Kiểm tra danh tính nguồn, số tệp, `data.yaml`, bảng lớp gốc và vẽ lưới ảnh mẫu thật có nhãn gốc. |
| **Mốc B** | Chạy mẫu thử (`SO_ANH_THU = 300`, có `SEED`) để kiểm chứng logic và kiểm tra bằng mắt trước khi chạy toàn nguồn. |
| **Mốc C** | Chạy toàn bộ nguồn (`SO_ANH_THU = 0`), lập thống kê, biểu đồ, rà trùng và tạo bảng ảnh đối chiếu `Ra_soat/`. |
| **Mốc D & E** | Sau khi xác nhận bảng lớp (`XAC_NHAN_BANG_LOP_GOC = True`) và rà soát, bật `XUAT_BO_DU_LIEU = True` để xuất `yolo2_char_num`, ảnh đại diện, `Ban_giao_OCR/` và chạy kiểm chứng nạp sau giải nén. |


In [ ]:
# @title Kết nối Drive và cấu hình chung YOLO2
from pathlib import Path
import io
import os
import shutil

# --- 1. Tham số điều khiển chính (đặt lên đầu để luôn được khởi tạo kể cả khi mount lỗi) ---
# Đặt sẵn "xu_ly_du_lieu" để chạy Mốc A & Mốc B ngay; sau khi xuất xong Mốc E, đổi về "xem_bao_cao" để bàn giao.
CHE_DO = "xu_ly_du_lieu"  # @param ["xu_ly_du_lieu", "xem_bao_cao"]
PHIEN_BAN_XUAT = "Ban_01"  # @param {type:"string"}
SO_ANH_THU = 300  # @param {type:"integer"}
# 300: mẫu thử Mốc B (lưu riêng ở Thu_300_anh); 0: toàn bộ nguồn Roboflow (lưu ở PHIEN_BAN_XUAT).
SEED = 42  # @param {type:"integer"}

# --- 2. Đường dẫn Drive / cục bộ ---
# Link thư mục dự án: https://drive.google.com/drive/u/1/folders/1nBzyrK4yzCwul_6WSiItAKXqjl21DEw6
ID_THU_MUC_DU_AN_DRIVE = "1nBzyrK4yzCwul_6WSiItAKXqjl21DEw6"
ID_THU_MUC_NGUON_DRIVE = "1Qm4crpxPTgGh5tLeGMWKAzR10inSxolC"
THU_MUC_DU_AN = "/content/drive/MyDrive/ML/Do_an_bien_so"  # @param {type:"string"}
# Để trống để tự tìm trong Drive (kể cả thư mục "Được chia sẻ với tôi" qua Drive API)
TEP_ROBOFLOW = ""  # @param {type:"string"}
THU_MUC_TAM_SSD = "/content/yolo2_tam"  # @param {type:"string"}

# --- 3. Tối ưu I/O và tiến độ ---
SO_LUONG = 2  # @param {type:"integer"}
KICH_THUOC_PHAN = 2000  # @param {type:"integer"}

# --- 4. Cổng an toàn cho Mốc D & E ---
XAC_NHAN_BANG_LOP_GOC = False  # @param {type:"boolean"}
XUAT_BO_DU_LIEU = False  # @param {type:"boolean"}
NEN_ZIP_XUAT = True  # @param {type:"boolean"}
CHO_PHEP_XUAT_MAU_THU = False  # @param {type:"boolean"}
GHI_DE_BAN_XUAT = False  # @param {type:"boolean"}

# --- 5. Ảnh đại diện và quy tắc nhóm đánh giá ---
SO_KICH_THUOC_DAI_DIEN = 5  # @param {type:"integer"}
SO_ANH_MOI_KICH_THUOC = 3  # @param {type:"integer"}
PHAM_VI_ANH_DAI_DIEN = "bo_giu_lai"  # @param ["bo_giu_lai", "nguon_ban_dau"]
CHI_MOT_DAI_DIEN_VAL_TEST = True  # @param {type:"boolean"}


def _y2_kiem_tra_quyen_ghi(folder):
    try:
        p = Path(folder)
        p.mkdir(parents=True, exist_ok=True)
        probe = p / ".y2_write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink(missing_ok=True)
        return True
    except OSError:
        return False


def _y2_tai_zip_tu_drive_api(dest_zip):
    """Nếu thư mục dự án nằm ở 'Được chia sẻ với tôi' (chưa tạo lối tắt vào MyDrive), tải thẳng từ ID thư mục được cấp quyền."""
    dest_zip = Path(dest_zip)
    if dest_zip.is_file() and dest_zip.stat().st_size > 1024 * 1024:
        return str(dest_zip), "Đã có sẵn tệp nguồn trên SSD tạm"
    try:
        from google.colab import auth  # type: ignore
        from googleapiclient.discovery import build  # type: ignore
        from googleapiclient.http import MediaIoBaseDownload  # type: ignore
        print("Đang xác thực quyền đọc Google Drive API cho thư mục được chia sẻ...", flush=True)
        auth.authenticate_user()
        service = build("drive", "v3", cache_discovery=False)
        res = service.files().list(
            q="name = 'roboflow_ocr_v1_yolov8.zip' and trashed = false",
            spaces="drive",
            fields="files(id, name, size, parents)",
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
            pageSize=10,
        ).execute()
        files = res.get("files", [])
        if not files:
            return "", "Tài khoản Google hiện tại chưa thấy tệp 'roboflow_ocr_v1_yolov8.zip' trên Drive."
        chosen = files[0]
        dest_zip.parent.mkdir(parents=True, exist_ok=True)
        tmp_zip = dest_zip.with_suffix(".part")
        print(f"Đang tải '{chosen['name']}' (ID: {chosen['id']}) từ Drive được chia sẻ về SSD {dest_zip}...", flush=True)
        request = service.files().get_media(fileId=chosen["id"], supportsAllDrives=True)
        with tmp_zip.open("wb") as f:
            downloader = MediaIoBaseDownload(f, request, chunksize=32 * 1024 * 1024)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"  Tiến độ tải nguồn: {int(status.progress() * 100)}%", flush=True)
        tmp_zip.replace(dest_zip)
        return str(dest_zip), f"Đã lấy từ Google Drive API (fileId={chosen['id']})"
    except Exception as e:
        return "", f"Không đọc được qua Drive API: {type(e).__name__}: {e}"


# Tự kết nối Google Drive an toàn (bắt mọi ngoại lệ để không làm gãy ô cấu hình khi chạy từ VS Code)
DANG_CHAY_COLAB = False
LOI_MOUNT_DRIVE = ""
try:
    import google.colab  # type: ignore
    DANG_CHAY_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
except Exception as _e:
    LOI_MOUNT_DRIVE = f"{type(_e).__name__}: {_e}"

# Tự động tìm thư mục dự án và tệp roboflow_ocr_v1_yolov8.zip
_ung_vien_du_an = [
    Path(THU_MUC_DU_AN),
    Path("/content/drive/MyDrive/ML/Do_an_bien_so"),
    Path("/content/drive/MyDrive/Do_an_bien_so"),
    Path(f"/content/drive/.shortcut-targets-by-id/{ID_THU_MUC_DU_AN_DRIVE}/Do_an_bien_so"),
    Path(f"/content/drive/.shortcut-targets-by-id/{ID_THU_MUC_DU_AN_DRIVE}"),
]
for _p in _ung_vien_du_an:
    if _p.is_dir():
        THU_MUC_DU_AN = str(_p)
        break

_ung_vien_zip = []
if TEP_ROBOFLOW.strip():
    _ung_vien_zip.append(Path(TEP_ROBOFLOW.strip()))
for _p in _ung_vien_du_an:
    _ung_vien_zip.append(_p / "01_Du_lieu_goc/Roboflow/roboflow_ocr_v1_yolov8.zip")
    _ung_vien_zip.append(_p / "Roboflow/roboflow_ocr_v1_yolov8.zip")
    _ung_vien_zip.append(_p / "roboflow_ocr_v1_yolov8.zip")
_ung_vien_zip.append(Path(f"/content/drive/.shortcut-targets-by-id/{ID_THU_MUC_NGUON_DRIVE}/Roboflow/roboflow_ocr_v1_yolov8.zip"))
_ung_vien_zip.append(Path(THU_MUC_TAM_SSD) / "roboflow_ocr_v1_yolov8.zip")

NGUON_ROBOFLOW = ""
for _zp in _ung_vien_zip:
    if _zp.is_file():
        NGUON_ROBOFLOW = str(_zp)
        break

# Nếu chưa thấy ở các đường dẫn cố định, quét nhanh trong MyDrive (tối đa 4 cấp)
if not NGUON_ROBOFLOW and Path("/content/drive/MyDrive").is_dir():
    for _found in Path("/content/drive/MyDrive").glob("**/roboflow_ocr_v1_yolov8.zip"):
        if _found.is_file():
            NGUON_ROBOFLOW = str(_found)
            break

# Nếu vẫn chưa thấy (ví dụ thư mục nằm trong "Được chia sẻ với tôi" chưa tạo lối tắt, hoặc chạy từ VS Code Colab), dùng Drive API
GHI_CHU_NGUON = "Tìm thấy trực tiếp trên ổ đĩa / Drive đã mount"
if not NGUON_ROBOFLOW and DANG_CHAY_COLAB:
    _api_zip, _api_msg = _y2_tai_zip_tu_drive_api(Path(THU_MUC_TAM_SSD) / "roboflow_ocr_v1_yolov8.zip")
    if _api_zip:
        NGUON_ROBOFLOW = _api_zip
    GHI_CHU_NGUON = _api_msg

if not NGUON_ROBOFLOW:
    NGUON_ROBOFLOW = TEP_ROBOFLOW.strip() or str(
        Path(THU_MUC_DU_AN) / "01_Du_lieu_goc/Roboflow/roboflow_ocr_v1_yolov8.zip"
    )

# Đảm bảo thư mục lưu báo cáo / kết quả có quyền ghi (nếu thư mục chia sẻ chỉ cho đọc thì lưu vào MyDrive hoặc /content)
if DANG_CHAY_COLAB and not _y2_kiem_tra_quyen_ghi(Path(THU_MUC_DU_AN) / "03_Bao_cao/03_YOLO2"):
    _fallback_drive = Path("/content/drive/MyDrive/ML/Do_an_bien_so")
    if Path("/content/drive/MyDrive").is_dir() and _y2_kiem_tra_quyen_ghi(_fallback_drive / "03_Bao_cao/03_YOLO2"):
        THU_MUC_DU_AN = str(_fallback_drive)
    else:
        THU_MUC_DU_AN = "/content/Do_an_bien_so"
        Path(THU_MUC_DU_AN).mkdir(parents=True, exist_ok=True)

PHAM_VI_TEN = PHIEN_BAN_XUAT if SO_ANH_THU == 0 else f"Thu_{SO_ANH_THU}_anh"

print("Môi trường:", "Google Colab" if DANG_CHAY_COLAB else "Cục bộ")
if LOI_MOUNT_DRIVE:
    print("Lưu ý mount Drive FUSE:", LOI_MOUNT_DRIVE, "(Nếu chạy qua VS Code Colab extension, hãy mở trực tiếp trên trình duyệt https://colab.research.google.com để mount Drive ổn định nhất)")
print("Chế độ:", "Xử lý dữ liệu và lưu báo cáo" if CHE_DO == "xu_ly_du_lieu" else "Xem báo cáo đã lưu (không quét nguồn)")
print("Phạm vi làm việc:", f"Toàn bộ nguồn ({PHIEN_BAN_XUAT})" if SO_ANH_THU == 0 else f"Mẫu thử {SO_ANH_THU} ảnh ({PHAM_VI_TEN})")
print("Thư mục lưu kết quả dự án:", THU_MUC_DU_AN)
print("Tệp nguồn Roboflow:", NGUON_ROBOFLOW, "—", "ĐÃ TÌM THẤY" if Path(NGUON_ROBOFLOW).is_file() else f"CHƯA THẤY ({GHI_CHU_NGUON})")


In [ ]:
# @title Hàm hỗ trợ Phần 1: đọc nguồn, kiểm tra ảnh–nhãn và lưu tiến độ — có thể thu gọn
"""Lõi Mốc A–C cho YOLO2 (chỉ đọc Roboflow OCR v1; bảo toàn nhãn ký tự gốc cho OCR)."""
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from pathlib import Path, PurePosixPath
from urllib.parse import urlsplit
from zipfile import ZipFile
import csv
import gzip
import hashlib
import io
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
import uuid

import numpy as np
import PIL
from PIL import Image, ImageDraw, ImageFont, features
import yaml

Y2_VERSION = "yolo2_1.0.0"
Y2_SCHEMA_VERSION = "1.0"
Y2_SOURCE_NAME = "Roboflow"
Y2_EXPECTED_PROJECT = "licence-plate-vn/ocr-stypd"
Y2_EXPECTED_VERSION = 1
Y2_BENCHMARK_COUNTS = {"train": 16209, "val": 2027, "test": 2026, "tong": 20262}
Y2_OCR_32_CHARS = "0123456789ABCDEFGHKLMNPQRSTUVXYZ"
Y2_OCR_32_SET = set(Y2_OCR_32_CHARS)
Y2_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
Y2_SPLIT_MAP = {"train": "train", "valid": "val", "val": "val", "test": "test"}
Y2_EPS_BOUNDS = 1e-6
Y2_HIGH_IOU_THRESHOLD = 0.85
Y2_PRIORITY = {"train": 0, "val": 1, "test": 2}

VN_TZ = timezone(timedelta(hours=7))


def y2_now_iso():
    return datetime.now(VN_TZ).isoformat(timespec="seconds")


def y2_decode_versions():
    return {
        "Python": platform.python_version(),
        "Pillow": PIL.__version__,
        "numpy": np.__version__,
        "PyYAML": getattr(yaml, "__version__", "unknown"),
        "JPEG": getattr(Image.core, "jpeglib_version", None),
        "libjpeg_turbo": features.version("libjpeg_turbo"),
    }


def y2_write_text_atomic(path, text, encoding="utf-8"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f"{path.name}.{uuid.uuid4().hex}.part")
    try:
        tmp.write_text(text, encoding=encoding)
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)


def y2_write_json(path, obj):
    y2_write_text_atomic(
        path,
        json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def y2_format_csv_val(val):
    if isinstance(val, bool):
        return "true" if val else "false"
    if val is None:
        return ""
    if isinstance(val, (list, dict)):
        return json.dumps(val, ensure_ascii=False)
    return str(val)


def y2_parse_bool(val):
    if isinstance(val, bool):
        return val
    s = str(val).strip().lower()
    if s == "true":
        return True
    if s == "false":
        return False
    raise ValueError(f"Giá trị boolean không hợp lệ: {val!r}")


def y2_write_csv(path, rows, fields=None):
    rows = list(rows)
    if not fields:
        fields = list(rows[0].keys()) if rows else ["ghi_chu"]
    buf = io.StringIO(newline="")
    writer = csv.DictWriter(buf, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        writer.writerow({k: y2_format_csv_val(row.get(k, "")) for k in fields})
    y2_write_text_atomic(path, buf.getvalue(), encoding="utf-8-sig")


def y2_read_csv(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def y2_display_table(rows, col_map=None, max_rows=None):
    rows = list(rows)
    if max_rows is not None:
        rows = rows[:max_rows]
    if not rows:
        print("(Bảng trống)")
        return
    try:
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(rows)
        if col_map:
            keep = [k for k in col_map if k in df.columns]
            df = df[keep].rename(columns=col_map) if keep else df.rename(columns=col_map)
        with pd.option_context("display.max_colwidth", 120):
            display(df)
    except ImportError:
        keys = [k for k in (col_map or rows[0]) if k in rows[0]]
        headers = [col_map[k] if col_map else k for k in keys]
        print(" | ".join(headers))
        print("-" * max(40, len(" | ".join(headers))))
        for r in rows:
            print(" | ".join(str(r.get(k, "")) for k in keys))


def y2_show_markdown(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(text))
    except ImportError:
        print(text)


def y2_show_image(path, width=1050):
    path = Path(path)
    if not path.is_file():
        print(f"Chưa thấy tệp hình: {path}")
        return
    try:
        from IPython.display import display, Image as DisplayImage
        display(DisplayImage(filename=str(path), width=width))
    except ImportError:
        print(f"[Hình đã lưu tại: {path}]")


def y2_report_dir(project, phien_ban_xuat, limit):
    scope = phien_ban_xuat if limit == 0 else f"Thu_{limit}_anh"
    return Path(project) / "03_Bao_cao/03_YOLO2" / scope, scope


def _y2_detect_split_and_kind(parts):
    """Nhận diện (prefix, split, kind, rel_after) theo cả hai bố cục <split>/images và images/<split>."""
    matches = []
    low = [p.lower() for p in parts]
    for i in range(len(low) - 1):
        if low[i] in Y2_SPLIT_MAP and low[i + 1] in {"images", "labels"}:
            matches.append((i, Y2_SPLIT_MAP[low[i]], low[i + 1], "/".join(parts[:i]), "/".join(parts[i + 2:])))
        elif low[i] in {"images", "labels"} and low[i + 1] in Y2_SPLIT_MAP:
            matches.append((i, Y2_SPLIT_MAP[low[i + 1]], low[i], "/".join(parts[:i]), "/".join(parts[i + 2:])))
    return matches


def y2_index_zip(z):
    """Lập chỉ mục ảnh–nhãn theo đường dẫn tương đối đầy đủ; bảo vệ đường dẫn archive."""
    infos = [i for i in z.infolist() if not i.is_dir()]
    names = [i.filename for i in infos]
    if len(names) != len(set(names)):
        raise ValueError("ZIP nguồn có tên tệp lặp; dừng để tránh đọc nhầm thành viên.")
    images, labels, other_images, yaml_files, readmes = [], {}, [], [], []
    for info in infos:
        fname = info.filename
        p = PurePosixPath(fname)
        if p.is_absolute() or ".." in p.parts or "\\" in fname:
            raise ValueError(f"Đường dẫn không an toàn trong ZIP: {fname}")
        if any(part.startswith(".") or part == "__MACOSX" for part in p.parts):
            continue
        ext = p.suffix.lower()
        name_low = p.name.lower()
        if name_low in {"data.yaml", "data.yml"}:
            yaml_files.append(info)
        if name_low.startswith("readme") and info.file_size <= 256 * 1024 and len(readmes) < 12:
            readmes.append({"tep": fname, "noi_dung": z.read(info).decode("utf-8-sig", errors="replace")})
        matches = _y2_detect_split_and_kind(p.parts[:-1])
        if len(matches) != 1:
            if ext in Y2_IMG_EXT:
                other_images.append(fname)
            continue
        _, split, kind, prefix, rel_dir = matches[0]
        rel_file = f"{rel_dir}/{p.name}" if rel_dir else p.name
        rel_stem = str(PurePosixPath(rel_file).with_suffix(""))
        pair_key = (prefix, split, rel_stem)
        if kind == "images" and ext in Y2_IMG_EXT:
            images.append({
                "source_member": fname,
                "original_split": split,
                "pair_key": pair_key,
                "file_size": info.file_size,
                "crc": info.CRC,
            })
        elif kind == "labels" and ext == ".txt":
            if pair_key in labels:
                raise ValueError(f"Có nhiều tệp nhãn cho cùng đường dẫn tương đối: {fname}")
            labels[pair_key] = fname
    pair_keys = [x["pair_key"] for x in images]
    if len(pair_keys) != len(set(pair_keys)):
        raise ValueError("Có nhiều tệp ảnh trùng đường dẫn gốc (khác phần mở rộng); không tự đoán ghép nhãn.")
    for item in images:
        item["label_member"] = labels.get(item.pop("pair_key"), "")
    used_labels = {x["label_member"] for x in images if x["label_member"]}
    orphan_labels = sorted(set(labels.values()) - used_labels)
    images.sort(key=lambda x: (x["original_split"], x["source_member"]))

    # Dấu metadata toàn ZIP (khóa cache khi tệp thay đổi mà không phải băm toàn bộ byte nếu không cần)
    meta = hashlib.sha256()
    for info in sorted(infos, key=lambda x: x.filename):
        meta.update(json.dumps([info.filename, info.file_size, info.compress_size, info.CRC], ensure_ascii=False).encode())
    return {
        "images": images,
        "orphan_labels": orphan_labels,
        "other_images": sorted(other_images),
        "yaml_files": yaml_files,
        "readmes": readmes,
        "metadata_zip": meta.hexdigest(),
    }


def y2_verify_identity(readmes):
    """Xác minh danh tính Roboflow từ README.dataset.txt / README.roboflow.txt; không suy từ tên ZIP."""
    urls = set()
    header_versions = set()
    for item in readmes:
        text = item["noi_dung"]
        urls.update(re.findall(r"https?://universe\.roboflow\.com/[^\s<>\"']+", text))
        if PurePosixPath(item["tep"]).name.lower() == "readme.roboflow.txt":
            first = next((ln.strip() for ln in text.splitlines() if ln.strip()), "")
            m = re.fullmatch(r".+?\s+-\s+v([0-9]+)(?:\s+.*)?", first)
            if m:
                header_versions.add(int(m.group(1)))
    project_urls, url_versions, exact_url = [], set(), ""
    for url in sorted(urls):
        parsed = urlsplit(url.rstrip(".,)]}"))
        if parsed.hostname != "universe.roboflow.com":
            continue
        path_part = parsed.path.strip("/")
        if path_part == Y2_EXPECTED_PROJECT:
            project_urls.append(url)
        m = re.fullmatch(re.escape(Y2_EXPECTED_PROJECT) + r"/(?:dataset/)?([0-9]+)", path_part)
        if m:
            ver = int(m.group(1))
            url_versions.add(ver)
            if ver == Y2_EXPECTED_VERSION:
                exact_url = url
    all_versions = url_versions | header_versions
    conflict = any(v != Y2_EXPECTED_VERSION for v in all_versions)
    if not conflict and (exact_url or (project_urls and header_versions == {Y2_EXPECTED_VERSION})):
        status = "khop_nguon_va_phien_ban"
        evidence = exact_url or f"{project_urls[0]} ; README.roboflow.txt ghi v{Y2_EXPECTED_VERSION}"
        verified_ver = str(Y2_EXPECTED_VERSION)
    elif conflict:
        status = "mau_thuan_phien_ban_hoac_du_an"
        evidence = "; ".join(sorted(urls))
        verified_ver = "mau_thuan"
    else:
        status = "chua_xac_minh"
        evidence = "; ".join(sorted(urls)) or "Khong tim thay lien ket Roboflow trong README"
        verified_ver = "chua_xac_minh"
    return {
        "trang_thai": status,
        "du_an_ky_vong": Y2_EXPECTED_PROJECT,
        "phien_ban_ky_vong": Y2_EXPECTED_VERSION,
        "phien_ban_xac_minh": verified_ver,
        "bang_chung": evidence,
        "so_tep_readme": len(readmes),
    }


def y2_parse_data_yaml(z, yaml_files):
    """Đọc data.yaml và bảo toàn đúng bảng lớp gốc dưới dạng chuỗi; cảnh báo nếu có giá trị bất thường."""
    if len(yaml_files) != 1:
        raise ValueError(f"Cần đúng 1 tệp data.yaml trong ZIP; tìm thấy {len(yaml_files)}: {[f.filename for f in yaml_files]}")
    info = yaml_files[0]
    if info.file_size > 1024 * 1024:
        raise ValueError("data.yaml lớn hơn 1 MiB; dừng kiểm tra.")
    raw_text = z.read(info).decode("utf-8-sig")
    cfg = yaml.safe_load(raw_text)
    if not isinstance(cfg, dict) or "names" not in cfg:
        raise ValueError("data.yaml không có trường names hợp lệ.")
    raw_names = cfg["names"]
    classes = {}
    if isinstance(raw_names, list):
        for idx, val in enumerate(raw_names):
            if isinstance(val, bool):
                raise ValueError(f"Lớp ID {idx} trong data.yaml bị YAML đọc thành boolean ({val}); cần kiểm tra raw data.yaml.")
            classes[idx] = str(val)
    elif isinstance(raw_names, dict):
        for k, val in raw_names.items():
            idx = int(k)
            if isinstance(val, bool):
                raise ValueError(f"Lớp ID {idx} trong data.yaml bị YAML đọc thành boolean ({val}); cần kiểm tra raw data.yaml.")
            classes[idx] = str(val)
        expected_ids = list(range(len(classes)))
        if sorted(classes.keys()) != expected_ids:
            raise ValueError(f"ID lớp trong data.yaml không liên tục từ 0: {sorted(classes.keys())}")
    else:
        raise ValueError("Trường names trong data.yaml phải là list hoặc dict.")
    nc = int(cfg.get("nc", len(classes)))
    if nc != len(classes):
        raise ValueError(f"Mâu thuẫn data.yaml: nc={nc} nhưng len(names)={len(classes)}.")
    rows = []
    for idx in range(nc):
        raw_name = classes[idx]
        char_val = raw_name.strip()
        rows.append({
            "original_class_id": idx,
            "raw_name": raw_name,
            "character": char_val,
            "in_ocr_32": char_val in Y2_OCR_32_SET,
        })
    return {
        "yaml_member": info.filename,
        "nc": nc,
        "classes": rows,
        "id_to_char": {r["original_class_id"]: r["character"] for r in rows},
        "raw_yaml_preview": raw_text[:4000],
    }


def y2_make_ids(source_member):
    """Tạo source_image_id và parent_id ổn định có namespace nguồn (không chỉ dùng basename)."""
    source_image_id = f"{Y2_SOURCE_NAME}::{source_member}"
    p = PurePosixPath(source_member)
    # Roboflow nối hậu tố .rf.<hex> cho biến thể cùng ảnh gốc qua các tập
    if re.search(r"\.rf\.[0-9a-fA-F]{16,}$", p.stem):
        base = p.stem.rsplit(".rf.", 1)[0]
        parent_id = f"{Y2_SOURCE_NAME}::rf::{base}"
        has_rf_parent = True
    else:
        parent_id = f"{Y2_SOURCE_NAME}::file::{source_member}"
        has_rf_parent = False
    return source_image_id, parent_id, has_rf_parent


def y2_box_iou(a, b):
    """Tính IoU cho hai hộp dạng (x1, y1, x2, y2)."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0.0:
        return 0.0
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    denom = area_a + area_b - inter
    return inter / denom if denom > 0.0 else 0.0


def y2_parse_label_text(text, id_to_char, width_px, height_px, source_image_id, eps=Y2_EPS_BOUNDS):
    """Đọc toàn bộ dòng nhãn gốc (đánh số dòng từ 0), bảo toàn original_class_id và trạng thái từng khung."""
    n_classes = len(id_to_char)
    boxes = []
    seen_exact = set()
    dup_line_count = 0
    for line_idx, raw_line in enumerate(text.splitlines()):
        if not raw_line.strip():
            continue
        box_id = f"{source_image_id}#L{line_idx}"
        parts = raw_line.split()
        reasons = []
        orig_cls = ""
        char_val = ""
        cx = cy = bw = bh = None
        x1 = y1 = x2 = y2 = None
        overshoot_px = 0.0
        if len(parts) != 5:
            reasons.append("can_5_cot")
        else:
            try:
                vals = [float(v) for v in parts]
            except ValueError:
                vals = None
                reasons.append("khong_phai_so")
            if vals is not None:
                if not all(math.isfinite(v) for v in vals):
                    reasons.append("so_khong_huu_han")
                else:
                    cls_f, cx, cy, bw, bh = vals
                    if cls_f != int(cls_f) or not (0 <= int(cls_f) < n_classes):
                        reasons.append("lop_khong_hop_le")
                        orig_cls = parts[0]
                    else:
                        orig_cls = int(cls_f)
                        char_val = id_to_char.get(orig_cls, "")
                    if bw <= 0.0 or bh <= 0.0:
                        reasons.append("rong_cao_khong_duong")
                    xn1, yn1 = cx - bw / 2.0, cy - bh / 2.0
                    xn2, yn2 = cx + bw / 2.0, cy + bh / 2.0
                    if min(xn1, yn1) < -eps or max(xn2, yn2) > 1.0 + eps:
                        reasons.append("vuot_bien_anh")
                    if width_px and height_px:
                        # Quy ước tọa độ XYXY theo ảnh gốc, cạnh phải/dưới nửa mở [x1, x2) x [y1, y2)
                        x1 = round(xn1 * width_px, 4)
                        y1 = round(yn1 * height_px, 4)
                        x2 = round(xn2 * width_px, 4)
                        y2 = round(yn2 * height_px, 4)
                        overshoot_px = round(
                            max(0.0, -xn1 * width_px, -yn1 * height_px, (xn2 - 1.0) * width_px, (yn2 - 1.0) * height_px),
                            4,
                        )
        is_exact_dup = False
        if not reasons:
            key = (orig_cls, round(cx, 7), round(cy, 7), round(bw, 7), round(bh, 7))
            if key in seen_exact:
                is_exact_dup = True
                dup_line_count += 1
                reasons.append("khung_lap_trung")
            seen_exact.add(key)
        bbox_status = "hop_le" if not reasons else ("khung_lap_trung" if reasons == ["khung_lap_trung"] else "khung_loi")
        boxes.append({
            "source_box_id": box_id,
            "source_line_index": line_idx,
            "original_class_id": orig_cls,
            "character": char_val,
            "raw_yolo_cx": cx,
            "raw_yolo_cy": cy,
            "raw_yolo_w": bw,
            "raw_yolo_h": bh,
            "x1_px": x1,
            "y1_px": y1,
            "x2_px": x2,
            "y2_px": y2,
            "overshoot_px": overshoot_px,
            "bbox_status": bbox_status,
            "reason_codes": reasons,
            "is_exact_dup": is_exact_dup,
        })

    # Đếm cặp khung chồng lấn IoU cao (gợi ý xem lại, KHÔNG tự xóa)
    high_iou_pairs = []
    valid_norm = [
        (b["source_line_index"], b["original_class_id"], (b["raw_yolo_cx"] - b["raw_yolo_w"] / 2.0,
         b["raw_yolo_cy"] - b["raw_yolo_h"] / 2.0, b["raw_yolo_cx"] + b["raw_yolo_w"] / 2.0,
         b["raw_yolo_cy"] + b["raw_yolo_h"] / 2.0))
        for b in boxes if b["bbox_status"] == "hop_le"
    ]
    for i in range(len(valid_norm)):
        for j in range(i + 1, len(valid_norm)):
            iou = y2_box_iou(valid_norm[i][2], valid_norm[j][2])
            if iou >= Y2_HIGH_IOU_THRESHOLD:
                high_iou_pairs.append({
                    "line_a": valid_norm[i][0],
                    "cls_a": valid_norm[i][1],
                    "line_b": valid_norm[j][0],
                    "cls_b": valid_norm[j][1],
                    "iou": round(iou, 4),
                })
    return boxes, dup_line_count, high_iou_pairs


def y2_image_quality(rgb_image):
    gray = rgb_image.convert("L")
    gray.thumbnail((256, 256))
    arr = np.asarray(gray, dtype=np.float32)
    if min(arr.shape) >= 3:
        lap = arr[:-2, 1:-1] + arr[2:, 1:-1] + arr[1:-1, :-2] + arr[1:-1, 2:] - 4.0 * arr[1:-1, 1:-1]
        sharp = float(lap.var())
    else:
        sharp = 0.0
    return {
        "sang": round(float(arr.mean()), 4),
        "tuong_phan": round(float(arr.std()), 4),
        "net": round(sharp, 4),
    }


def y2_process_one(zip_path_or_obj, item, id_to_char):
    """Giải mã 1 ảnh và toàn bộ dòng nhãn gốc; không làm hỏng batch nếu 1 ảnh lỗi."""
    source_image_id, parent_id, has_rf_parent = y2_make_ids(item["source_member"])
    row = {
        "source": Y2_SOURCE_NAME,
        "source_member": item["source_member"],
        "label_member": item["label_member"],
        "source_image_id": source_image_id,
        "original_split": item["original_split"],
        "parent_id": parent_id,
        "has_rf_parent": has_rf_parent,
        "width_px": None,
        "height_px": None,
        "exif_orientation": 1,
        "pixel_sha256": "",
        "dhash": "",
        "image_error": "",
        "label_file_error": "",
        "quality": {},
        "boxes": [],
        "box_count_total": 0,
        "box_count_valid": 0,
        "box_count_invalid": 0,
        "dup_line_count": 0,
        "high_iou_pairs": [],
    }
    def _run_with_zip(z):
        # 1. Giải mã ảnh thật (kiểm tra CRC ZIP + pixel thực tế, không tự xoay EXIF)
        try:
            raw_img = z.read(item["source_member"])
            with Image.open(io.BytesIO(raw_img)) as im:
                im.load()
                row["exif_orientation"] = int(im.getexif().get(274, 1))
                rgb = im.convert("RGB")
            w, h = rgb.size
            row["width_px"], row["height_px"] = w, h
            row["pixel_sha256"] = hashlib.sha256(f"RGB:{w}:{h}:".encode() + rgb.tobytes()).hexdigest()
            small = np.asarray(rgb.convert("L").resize((9, 8)), dtype=np.int16)
            row["dhash"] = np.packbits(small[:, 1:] > small[:, :-1]).tobytes().hex()
            row["quality"] = y2_image_quality(rgb)
        except Exception as e:
            row["image_error"] = f"{type(e).__name__}: {str(e)[:200]}"
        # 2. Đọc tệp nhãn gốc
        try:
            if not item["label_member"]:
                row["label_file_error"] = "thieu_tep_nhan"
            elif z.getinfo(item["label_member"]).file_size > 2 * 1024 * 1024:
                row["label_file_error"] = "tep_nhan_qua_lon"
            else:
                label_text = z.read(item["label_member"]).decode("utf-8-sig")
                boxes, dup_cnt, hiou = y2_parse_label_text(
                    label_text, id_to_char, row["width_px"], row["height_px"], source_image_id
                )
                row["boxes"] = boxes
                row["box_count_total"] = len(boxes)
                row["box_count_valid"] = sum(b["bbox_status"] == "hop_le" for b in boxes)
                row["box_count_invalid"] = sum(b["bbox_status"] == "khung_loi" for b in boxes)
                row["dup_line_count"] = dup_cnt
                row["high_iou_pairs"] = hiou
        except (UnicodeError, OSError, ValueError) as e:
            row["label_file_error"] = f"khong_doc_duoc_nhan:{type(e).__name__}"

    if isinstance(zip_path_or_obj, ZipFile):
        _run_with_zip(zip_path_or_obj)
    else:
        with ZipFile(zip_path_or_obj) as z:
            _run_with_zip(z)
    return row


def _y2_load_font(size=13):
    for name in ["DejaVuSans.ttf", "arial.ttf"]:
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    return ImageFont.load_default()


def y2_render_labeled_grid(zip_path, sample_items, id_to_char, out_path, cols=3, title_fn=None):
    """Vẽ lưới ảnh biển số thật kèm khung và ID:ký tự gốc để đối chiếu trực quan."""
    if not sample_items:
        return ""
    font = _y2_load_font(12)
    tw, th = 400, 260
    nr = math.ceil(len(sample_items) / cols)
    canvas = Image.new("RGB", (cols * tw, nr * th), "white")
    draw = ImageDraw.Draw(canvas)
    palette = ["#00b84c", "#e65100", "#1565c0", "#8e24aa", "#c62828", "#00838f"]
    with ZipFile(zip_path) as z:
        for idx, item in enumerate(sample_items):
            ox, oy = (idx % cols) * tw + 8, (idx // cols) * th + 36
            try:
                raw = z.read(item["source_member"])
                with Image.open(io.BytesIO(raw)) as im:
                    rgb = im.convert("RGB")
                ow, oh = rgb.size
                scale = min((tw - 16) / max(1, ow), (th - 54) / max(1, oh))
                nw, nh = max(1, int(round(ow * scale))), max(1, int(round(oh * scale)))
                thumb = rgb.resize((nw, nh))
                canvas.paste(thumb, (ox, oy))
                boxes = item.get("boxes")
                if boxes is None and item.get("label_member"):
                    txt = z.read(item["label_member"]).decode("utf-8-sig", errors="replace")
                    boxes, _, _ = y2_parse_label_text(txt, id_to_char, ow, oh, item.get("source_image_id", ""))
                for b_i, b in enumerate(boxes or []):
                    if b.get("raw_yolo_cx") is None:
                        continue
                    cx, cy, bw, bh = b["raw_yolo_cx"], b["raw_yolo_cy"], b["raw_yolo_w"], b["raw_yolo_h"]
                    xa = max(0.0, min(ow, (cx - bw / 2.0) * ow)) * scale + ox
                    ya = max(0.0, min(oh, (cy - bh / 2.0) * oh)) * scale + oy
                    xb = max(0.0, min(ow, (cx + bw / 2.0) * ow)) * scale + ox
                    yb = max(0.0, min(oh, (cy + bh / 2.0) * oh)) * scale + oy
                    col = "#d50000" if b.get("bbox_status") == "khung_loi" else palette[b_i % len(palette)]
                    if xb >= xa and yb >= ya:
                        draw.rectangle((xa, ya, xb, yb), outline=col, width=2)
                        lbl = f"{b['original_class_id']}:{b['character'] or '?'}"
                        draw.text((xa, max(oy - 12, ya - 13)), lbl, fill=col, font=font)
            except Exception as e:
                draw.text((ox, oy + 20), f"Loi doc anh: {type(e).__name__}", fill="red", font=font)
            head = title_fn(item) if title_fn else f"{item['original_split']} | {PurePosixPath(item['source_member']).name[:42]}"
            draw.text((ox, oy - 30), head[:52], fill="black", font=font)
            sub = item.get("note", f"Khung: {len(item.get('boxes') or [])} | {item.get('width_px', '?')}x{item.get('height_px', '?')}")
            draw.text((ox, (idx // cols) * th + th - 16), str(sub)[:55], fill="#333333", font=font)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path, quality=94)
    return str(out_path)


def y2_step1_inspect_source(project, source, phien_ban_xuat="Ban_01", limit=300, mode="xem_bao_cao", seed=42):
    """Mốc A: kiểm tra danh tính, số tệp theo split, bảng lớp data.yaml và ảnh mẫu thật có nhãn gốc."""
    if mode not in {"xem_bao_cao", "xu_ly_du_lieu"}:
        raise ValueError("Chế độ không hợp lệ.")
    report, scope = y2_report_dir(project, phien_ban_xuat, limit)
    saved = report / "moc_a_kiem_tra_nguon.json"
    if mode == "xem_bao_cao":
        if not saved.is_file():
            return {"thong_bao": f"Chưa có báo cáo Mốc A cho phạm vi {scope}. Đổi CHE_DO='xu_ly_du_lieu' để chạy."}
        data = json.loads(saved.read_text(encoding="utf-8"))
        if data.get("phien_ban_ma") != Y2_VERSION or data.get("gioi_han") != limit:
            return {"thong_bao": "Báo cáo Mốc A đã lưu không khớp phiên bản mã hoặc phạm vi mẫu; cần chạy lại xử lý."}
        data["cach_doc"] = "Đọc báo cáo Mốc A đã lưu; không mở tệp ZIP nguồn."
        return data

    src_path = Path(source)
    if not src_path.is_file():
        raise FileNotFoundError(
            f"Chưa tìm thấy nguồn Roboflow tại: {src_path}. "
            "Kiểm tra quyền đọc Drive hoặc đường dẫn lối tắt; không tự tải nguồn khác thay thế."
        )
    t0 = time.monotonic()
    report.mkdir(parents=True, exist_ok=True)
    st = src_path.stat()
    with ZipFile(src_path) as z:
        idx = y2_index_zip(z)
        identity = y2_verify_identity(idx["readmes"])
        yaml_info = y2_parse_data_yaml(z, idx["yaml_files"])

    split_imgs = Counter(x["original_split"] for x in idx["images"])
    split_lbls = Counter(x["original_split"] for x in idx["images"] if x["label_member"])
    split_table = []
    for sp in ["train", "val", "test"]:
        n_img = split_imgs[sp]
        n_lbl = split_lbls[sp]
        bench = Y2_BENCHMARK_COUNTS[sp]
        split_table.append({
            "tap": sp,
            "so_tep_anh_nguon": n_img,
            "so_tep_nhan_khop": n_lbl,
            "moc_kiem_ke": bench,
            "chenh_lech_voi_moc": n_img - bench,
        })
    total_imgs = len(idx["images"])
    split_table.append({
        "tap": "tong",
        "so_tep_anh_nguon": total_imgs,
        "so_tep_nhan_khop": sum(split_lbls.values()),
        "moc_kiem_ke": Y2_BENCHMARK_COUNTS["tong"],
        "chenh_lech_voi_moc": total_imgs - Y2_BENCHMARK_COUNTS["tong"],
    })

    # Chọn 9 ảnh mẫu ngẫu nhiên cố định theo seed (phân bố các split) để vẽ đối chiếu Mốc A
    rng = random.Random(seed)
    sample_a = []
    for sp in ["train", "val", "test"]:
        pool = [x for x in idx["images"] if x["original_split"] == sp]
        if pool:
            k = min(3, len(pool))
            sample_a.extend(rng.sample(pool, k))
    sample_rows = []
    with ZipFile(src_path) as z:
        for item in sample_a:
            sample_rows.append(y2_process_one(z, item, yaml_info["id_to_char"]))
    grid_path = y2_render_labeled_grid(
        src_path,
        sample_rows,
        yaml_info["id_to_char"],
        report / "moc_a_doi_chieu_lop.png",
        cols=3,
    )

    y2_write_csv(report / "moc_a_kiem_ke_split.csv", split_table)
    y2_write_csv(report / "moc_a_bang_lop_goc.csv", yaml_info["classes"])
    y2_write_json(
        report / "bang_lop_goc_tam.json",
        {
            "schema_version": Y2_SCHEMA_VERSION,
            "source": Y2_SOURCE_NAME,
            "source_file": src_path.name,
            "source_identity": identity,
            "yaml_member": yaml_info["yaml_member"],
            "nc": yaml_info["nc"],
            "verification_status": "chua_xac_nhan_bang_mat",
            "classes": yaml_info["classes"],
        },
    )
    res = {
        "phien_ban_ma": Y2_VERSION,
        "schema_version": Y2_SCHEMA_VERSION,
        "thoi_diem": y2_now_iso(),
        "pham_vi": scope,
        "gioi_han": limit,
        "seed": seed,
        "nguon": str(src_path),
        "ten_tep_nguon": src_path.name,
        "dung_luong_MiB": round(st.st_size / (1024 * 1024), 2),
        "metadata_zip": idx["metadata_zip"],
        "thu_vien": y2_decode_versions(),
        "danh_tinh": identity,
        "bang_split": split_table,
        "so_nhan_mo_coi": len(idx["orphan_labels"]),
        "so_anh_ngoai_cau_truc": len(idx["other_images"]),
        "yaml_member": yaml_info["yaml_member"],
        "nc": yaml_info["nc"],
        "bang_lop_goc": yaml_info["classes"],
        "so_lop_ngoai_ocr32": sum(not r["in_ocr_32"] for r in yaml_info["classes"]),
        "anh_doi_chieu_moc_a": grid_path,
        "bao_cao": str(report),
        "thoi_gian_giay": round(time.monotonic() - t0, 3),
    }
    y2_write_json(saved, res)
    return res


def _y2_load_progress_parts(parts_dir, selected, batch_size):
    rows = []
    for start in range(0, len(selected), batch_size):
        part_path = parts_dir / f"phan_{start // batch_size:05d}.jsonl.gz"
        if not part_path.is_file():
            break
        with gzip.open(part_path, "rt", encoding="utf-8") as f:
            batch = [json.loads(line) for line in f]
        expect = selected[start : start + batch_size]
        if [r["source_member"] for r in batch] != [x["source_member"] for x in expect]:
            raise ValueError(f"Phần tiến độ {part_path.name} không khớp danh sách ảnh; dừng để tránh lệch mẫu.")
        rows.extend(batch)
    return rows


def y2_step2_scan_dataset(
    project,
    source,
    phien_ban_xuat="Ban_01",
    limit=300,
    mode="xem_bao_cao",
    seed=42,
    batch_size=2000,
    local_root="/content/yolo2_tam",
    workers=2,
    stop_after_batches=None,
):
    """Mốc B/C: quét ảnh–nhãn từng phần, hỗ trợ chạy tiếp, khóa cache phiên bản và đọc trên SSD."""
    if mode not in {"xem_bao_cao", "xu_ly_du_lieu"} or limit < 0 or batch_size < 1:
        raise ValueError("Tham số chế độ / giới hạn / kích thước phần không hợp lệ.")
    report, scope = y2_report_dir(project, phien_ban_xuat, limit)
    summary_path = report / "ket_qua_quet.json"
    if mode == "xem_bao_cao":
        if not summary_path.is_file():
            return {"thong_bao": f"Chưa có kết quả quét cho phạm vi {scope}. Đổi CHE_DO='xu_ly_du_lieu' để chạy."}
        res = json.loads(summary_path.read_text(encoding="utf-8"))
        if res.get("phien_ban_ma") != Y2_VERSION or res.get("gioi_han") != limit or res.get("seed") != seed:
            return {"thong_bao": "Kết quả quét đã lưu khác phiên bản mã/phạm vi/seed; cần chạy lại xử lý."}
        res["cach_doc"] = "Đọc kết quả quét đã lưu; không mở ZIP nguồn và không giải mã lại ảnh."
        return res

    workers = max(1, min(4, int(workers)))
    src_path = Path(source)
    if not src_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy nguồn Roboflow: {src_path}")
    t0 = time.monotonic()
    report.mkdir(parents=True, exist_ok=True)
    st_before = src_path.stat()
    with ZipFile(src_path) as z:
        idx = y2_index_zip(z)
        identity_src = y2_verify_identity(idx["readmes"])
        yaml_info = y2_parse_data_yaml(z, idx["yaml_files"])
    t_index = time.monotonic() - t0
    images = idx["images"]
    if not images:
        raise ValueError("Không tìm thấy ảnh hợp lệ trong cấu trúc train/val/test.")

    # Chọn mẫu xác định theo seed (phân bố đều qua các tập khi chạy mẫu thử)
    if limit > 0:
        ranked = sorted(
            images,
            key=lambda x: hashlib.sha256(f"{seed}::{x['source_member']}".encode()).hexdigest(),
        )
        selected = sorted(ranked[:limit], key=lambda x: (x["original_split"], x["source_member"]))
    else:
        selected = list(images)

    # Khóa cache theo nguồn + phiên bản mã + thư viện giải mã + quy tắc + phạm vi + seed
    cache_key = {
        "phien_ban_ma": Y2_VERSION,
        "schema_version": Y2_SCHEMA_VERSION,
        "thu_vien_giai_ma": y2_decode_versions(),
        "metadata_zip": idx["metadata_zip"],
        "source_file": src_path.name,
        "size": st_before.st_size,
        "mtime_ns": st_before.st_mtime_ns,
        "gioi_han": limit,
        "seed": seed,
        "kich_thuoc_phan": batch_size,
        "eps_bien": Y2_EPS_BOUNDS,
        "high_iou_threshold": Y2_HIGH_IOU_THRESHOLD,
    }
    parts_dir = report / "Tien_do"
    parts_dir.mkdir(parents=True, exist_ok=True)
    id_file = parts_dir / "nguon_va_quy_tac.json"
    if id_file.is_file():
        old_key = json.loads(id_file.read_text(encoding="utf-8"))
        if old_key != cache_key:
            raise ValueError(
                "Nguồn, phiên bản quy tắc hoặc thư viện giải mã đã thay đổi so với cache cũ. "
                "Không tự dùng tiến độ cũ; hãy đổi PHIEN_BAN_XUAT (ví dụ Ban_02) hoặc dọn thư mục cũ trước khi chạy mới."
            )
    else:
        y2_write_json(id_file, cache_key)

    rows = _y2_load_progress_parts(parts_dir, selected, batch_size)
    resumed_from = len(rows)
    print(f"[{scope}] Tổng mục tiêu: {len(selected):,}/{len(images):,} ảnh. Đã có trong tiến độ: {resumed_from:,}.", flush=True)

    # Quản lý SSD cache: mẫu nhỏ (<= 1000 ảnh) đọc trực tiếp; chạy lớn copy 1 lần sang SSD
    local_dir = Path(local_root)
    local_dir.mkdir(parents=True, exist_ok=True)
    direct_trial = limit > 0 and len(selected) <= 1000
    read_zip_path = src_path
    copied_sec = 0.0
    if len(rows) < len(selected) and not direct_trial and src_path.resolve() != (local_dir / src_path.name).resolve():
        cached_zip = local_dir / src_path.name
        mark_file = local_dir / "cache_nguon.json"
        ssd_key = {k: cache_key[k] for k in ["metadata_zip", "source_file", "size", "mtime_ns"]}
        cache_ok = (
            cached_zip.is_file()
            and mark_file.is_file()
            and cached_zip.stat().st_size == st_before.st_size
            and json.loads(mark_file.read_text(encoding="utf-8")) == ssd_key
        )
        if not cache_ok:
            need_bytes = st_before.st_size + 512 * 1024 * 1024
            if shutil.disk_usage(local_dir).free < need_bytes:
                print("Cảnh báo: ổ tạm SSD không đủ dung lượng dự phòng; đọc trực tiếp từ nguồn.", flush=True)
            else:
                tc = time.monotonic()
                tmp_zip = cached_zip.with_suffix(".part")
                shutil.copyfile(src_path, tmp_zip)
                st_after_cp = src_path.stat()
                if (st_after_cp.st_size, st_after_cp.st_mtime_ns) != (st_before.st_size, st_before.st_mtime_ns):
                    tmp_zip.unlink(missing_ok=True)
                    raise ValueError("Tệp nguồn thay đổi trong lúc sao chép sang SSD.")
                tmp_zip.replace(cached_zip)
                y2_write_json(mark_file, ssd_key)
                copied_sec = time.monotonic() - tc
                read_zip_path = cached_zip
                print(f"Đã sao chép ZIP nguồn sang SSD ({copied_sec:.2f}s).", flush=True)
        else:
            read_zip_path = cached_zip
            print("Dùng lại bản ZIP khớp phiên bản trên SSD.", flush=True)

    scan_t0 = time.monotonic()
    batches_done_this_run = 0
    if len(rows) < len(selected):
        for start in range(len(rows), len(selected), batch_size):
            if stop_after_batches is not None and batches_done_this_run >= stop_after_batches:
                break
            chunk = selected[start : start + batch_size]
            if workers == 1:
                with ZipFile(read_zip_path) as z:
                    batch = [y2_process_one(z, item, yaml_info["id_to_char"]) for item in chunk]
            else:
                # Mỗi luồng mở handle ZipFile riêng theo đường dẫn SSD — an toàn luồng và giữ đúng thứ tự
                with ThreadPoolExecutor(max_workers=workers) as pool:
                    batch = list(pool.map(lambda it: y2_process_one(read_zip_path, it, yaml_info["id_to_char"]), chunk))
            part_path = parts_dir / f"phan_{start // batch_size:05d}.jsonl.gz"
            local_tmp = local_dir / f"{part_path.name}.{uuid.uuid4().hex}.part"
            with gzip.open(local_tmp, "wt", encoding="utf-8") as f:
                for r in batch:
                    f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
            drive_tmp = part_path.with_name(f"{part_path.name}.{uuid.uuid4().hex}.part")
            shutil.copyfile(local_tmp, drive_tmp)
            local_tmp.unlink(missing_ok=True)
            drive_tmp.replace(part_path)
            rows.extend(batch)
            batches_done_this_run += 1
            print(f"Đã kiểm tra và lưu {len(rows):,}/{len(selected):,} ảnh.", flush=True)
    scan_sec = time.monotonic() - scan_t0

    if len(rows) < len(selected):
        return {
            "trang_thai": "tam_dung_giua_chung",
            "da_luu": len(rows),
            "muc_tieu": len(selected),
            "bao_cao": str(report),
        }

    st_end = src_path.stat()
    if (st_end.st_size, st_end.st_mtime_ns) != (st_before.st_size, st_before.st_mtime_ns):
        raise ValueError("Nguồn thay đổi trong khi quét; không công bố báo cáo hoàn tất.")

    # Ghi toàn bộ bản ghi chi tiết (đủ để vẽ lại mọi biểu đồ mà không đọc lại ảnh)
    detail_path = report / "chi_tiet_anh.jsonl.gz"
    detail_tmp = detail_path.with_name(f"{detail_path.name}.{uuid.uuid4().hex}.part")
    with gzip.open(detail_tmp, "wt", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    detail_tmp.replace(detail_path)
    y2_write_csv(report / "nhan_mo_coi.csv", [{"label_member": p} for p in idx["orphan_labels"]], ["label_member"])

    # Vẽ mẫu kiểm tra ảnh–nhãn (ngẫu nhiên cố định theo seed + lớp hiếm)
    rng = random.Random(seed + 1)
    valid_rows = [r for r in rows if not r["image_error"] and r["box_count_valid"] > 0]
    sample_vis = rng.sample(valid_rows, min(9, len(valid_rows))) if valid_rows else []
    sample_grid = y2_render_labeled_grid(
        read_zip_path, sample_vis, yaml_info["id_to_char"], report / "mau_anh_va_nhan_goc.png", cols=3
    )

    res = {
        "phien_ban_ma": Y2_VERSION,
        "schema_version": Y2_SCHEMA_VERSION,
        "thoi_diem": y2_now_iso(),
        "pham_vi": scope,
        "toan_bo": len(rows) == len(images),
        "gioi_han": limit,
        "seed": seed,
        "nguon": str(src_path),
        "ten_tep_nguon": src_path.name,
        "metadata_zip": idx["metadata_zip"],
        "danh_tinh": identity_src,
        "thu_vien_giai_ma": y2_decode_versions(),
        "bang_lop_goc": yaml_info["classes"],
        "so_anh_nguon": len(images),
        "so_anh_da_xet": len(rows),
        "so_anh_nguon_theo_tap": dict(Counter(x["original_split"] for x in images)),
        "so_nhan_mo_coi": len(idx["orphan_labels"]),
        "doc_tiep_tu": resumed_from,
        "so_luong": workers,
        "bao_cao": str(report),
        "anh_mau_kiem_tra": sample_grid,
        "thoi_gian_giay": {
            "kiem_ke": round(t_index, 3),
            "sao_chep_ssd": round(copied_sec, 3),
            "quet_anh_va_luu_tien_do": round(scan_sec, 3),
            "tong": round(time.monotonic() - t0, 3),
        },
    }
    y2_write_json(summary_path, res)
    return res


In [ ]:
# @title Hàm hỗ trợ Phần 2: thống kê, biểu đồ, nhóm trùng và bảng ảnh rà soát — có thể thu gọn
"""Phân tích đặc điểm dữ liệu, đối chiếu trùng lặp qua tập và lập bảng ảnh rà soát YOLO2."""
import matplotlib.pyplot as plt


def y2_load_detail_rows(report):
    with gzip.open(Path(report) / "chi_tiet_anh.jsonl.gz", "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def _y2_resolve_readable_zip(scan_result, local_root="/content/yolo2_tam"):
    """Chỉ dùng lại bản ZIP trên SSD khi dấu metadata_zip khớp đúng nguồn đã quét."""
    if local_root:
        local_dir = Path(local_root)
        cached = local_dir / scan_result["ten_tep_nguon"]
        mark = local_dir / "cache_nguon.json"
        if cached.is_file() and mark.is_file():
            try:
                if json.loads(mark.read_text(encoding="utf-8")).get("metadata_zip") == scan_result["metadata_zip"]:
                    return cached
            except (OSError, ValueError):
                pass
    return Path(scan_result["nguon"])


def _y2_box_signature(row):
    """Chữ ký bộ khung gốc (dùng phát hiện mâu thuẫn nhãn giữa hai ảnh giống hệt pixel)."""
    sig = []
    for b in row["boxes"]:
        if b["bbox_status"] == "hop_le":
            sig.append((
                b["original_class_id"],
                round(b["raw_yolo_cx"], 5),
                round(b["raw_yolo_cy"], 5),
                round(b["raw_yolo_w"], 5),
                round(b["raw_yolo_h"], 5),
            ))
        else:
            sig.append(("ERR", b["bbox_status"], tuple(b["reason_codes"])))
    sig.sort(key=str)
    return json.dumps(sig, ensure_ascii=False, separators=(",", ":"))


def y2_build_groups_and_edges(rows):
    """Phát hiện trùng pixel, họ tên gốc .rf. và gợi ý dHash; tạo group_id ổn định cho quan hệ đã xác nhận."""
    parent_idx = list(range(len(rows)))

    def find(i):
        while parent_idx[i] != i:
            parent_idx[i] = parent_idx[parent_idx[i]]
            i = parent_idx[i]
        return i

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent_idx[rb] = ra

    by_pixel = defaultdict(list)
    by_parent = defaultdict(list)
    by_dhash = defaultdict(list)
    for i, r in enumerate(rows):
        if r["pixel_sha256"]:
            by_pixel[r["pixel_sha256"]].append(i)
        if r["has_rf_parent"]:
            by_parent[r["parent_id"]].append(i)
        if r["dhash"]:
            by_dhash[r["dhash"]].append(i)

    edges = []
    pixel_conflict_images = set()
    group_summary = []

    # 1. Trùng toàn bộ pixel RGB (đã xác nhận)
    px_dups = [ids for ids in by_pixel.values() if len(ids) > 1]
    px_cross, px_conflicts = 0, 0
    for ids in px_dups:
        ids_s = sorted(ids, key=lambda i: rows[i]["source_image_id"])
        sigs = {_y2_box_signature(rows[i]) for i in ids_s}
        conflict = len(sigs) > 1
        if conflict:
            px_conflicts += 1
            for i in ids_s:
                pixel_conflict_images.add(rows[i]["source_image_id"])
        if len({rows[i]["original_split"] for i in ids_s}) > 1:
            px_cross += 1
        anchor = ids_s[0]
        for other in ids_s[1:]:
            union(anchor, other)
            edges.append({
                "edge_id": f"E_PX_{len(edges)+1:06d}",
                "source_image_id_a": rows[anchor]["source_image_id"],
                "source_image_id_b": rows[other]["source_image_id"],
                "split_a": rows[anchor]["original_split"],
                "split_b": rows[other]["original_split"],
                "evidence_type": "pixel_sha256",
                "confirmation_status": "da_xac_nhan",
                "label_conflict": conflict,
                "decision_code": "giu_ngoai_cho_doi_chieu_nhan" if conflict else "noi_nhom_xac_nhan",
                "note": "Giong toan bo pixel RGB va kich thuoc",
            })
    group_summary.append({
        "loai_quan_he": "pixel_sha256",
        "mo_ta": "Ảnh giống toàn bộ pixel RGB & kích thước",
        "muc_xac_nhan": "da_xac_nhan",
        "so_nhom": len(px_dups),
        "so_anh_trong_nhom": sum(len(g) for g in px_dups),
        "so_nhom_qua_nhieu_tap": px_cross,
        "so_nhom_mau_thuan_nhan": px_conflicts,
    })

    # 2. Biến thể cùng ảnh gốc Roboflow (.rf.) (đã xác nhận)
    rf_dups = [ids for ids in by_parent.values() if len(ids) > 1]
    rf_cross = 0
    for ids in rf_dups:
        ids_s = sorted(ids, key=lambda i: rows[i]["source_image_id"])
        if len({rows[i]["original_split"] for i in ids_s}) > 1:
            rf_cross += 1
        anchor = ids_s[0]
        for other in ids_s[1:]:
            union(anchor, other)
            edges.append({
                "edge_id": f"E_RF_{len(edges)+1:06d}",
                "source_image_id_a": rows[anchor]["source_image_id"],
                "source_image_id_b": rows[other]["source_image_id"],
                "split_a": rows[anchor]["original_split"],
                "split_b": rows[other]["original_split"],
                "evidence_type": "parent_id_rf",
                "confirmation_status": "da_xac_nhan",
                "label_conflict": False,
                "decision_code": "noi_nhom_xac_nhan",
                "note": "Cung ten goc truoc hau to .rf. cua Roboflow",
            })
    group_summary.append({
        "loai_quan_he": "parent_id_rf",
        "mo_ta": "Biến thể cùng ảnh gốc Roboflow (.rf.)",
        "muc_xac_nhan": "da_xac_nhan",
        "so_nhom": len(rf_dups),
        "so_anh_trong_nhom": sum(len(g) for g in rf_dups),
        "so_nhom_qua_nhieu_tap": rf_cross,
        "so_nhom_mau_thuan_nhan": 0,
    })

    # Gán group_id ổn định cho từng thành phần liên thông của các cạnh ĐÃ XÁC NHẬN
    comps = defaultdict(list)
    for i in range(len(rows)):
        comps[find(i)].append(i)
    confirmed_cross_groups = 0
    for ids in comps.values():
        canonical = min(rows[i]["parent_id"] for i in ids)
        gid = f"G::{canonical}"
        cross = len({rows[i]["original_split"] for i in ids}) > 1
        if cross:
            confirmed_cross_groups += 1
        for i in ids:
            rows[i]["group_id"] = gid
            rows[i]["group_cross_split"] = cross
            rows[i]["pixel_label_conflict"] = rows[i]["source_image_id"] in pixel_conflict_images

    # 3. Gợi ý gần giống dHash (KHÔNG nối tự động vào group_id; chỉ tạo cạnh chưa xác nhận)
    dh_dups = [ids for ids in by_dhash.values() if len(ids) > 1 and len({rows[i]["group_id"] for i in ids}) > 1]
    dh_cross = 0
    for ids in dh_dups:
        ids_s = sorted(ids, key=lambda i: rows[i]["source_image_id"])
        if len({rows[i]["original_split"] for i in ids_s}) > 1:
            dh_cross += 1
        anchor = ids_s[0]
        for other in ids_s[1:]:
            if rows[anchor]["group_id"] == rows[other]["group_id"]:
                continue
            edges.append({
                "edge_id": f"E_DH_{len(edges)+1:06d}",
                "source_image_id_a": rows[anchor]["source_image_id"],
                "source_image_id_b": rows[other]["source_image_id"],
                "split_a": rows[anchor]["original_split"],
                "split_b": rows[other]["original_split"],
                "evidence_type": "dhash_goi_y",
                "confirmation_status": "chua_xac_nhan",
                "label_conflict": False,
                "decision_code": "chi_goi_y_can_xem_anh",
                "note": "Trung dHash 64-bit; chua du ket luan trung neu chua doi chieu bang mat",
            })
    group_summary.append({
        "loai_quan_he": "dhash_goi_y",
        "mo_ta": "Gợi ý ảnh tương tự dHash (khác nhóm đã xác nhận)",
        "muc_xac_nhan": "chua_xac_nhan",
        "so_nhom": len(dh_dups),
        "so_anh_trong_nhom": sum(len(g) for g in dh_dups),
        "so_nhom_qua_nhieu_tap": dh_cross,
        "so_nhom_mau_thuan_nhan": 0,
    })
    return group_summary, edges, confirmed_cross_groups


def y2_step3_stats_and_charts(scan_result, mode="xem_bao_cao"):
    """Mục 3 & 4: tổng hợp toàn bộ bảng thống kê, lưu CSV và xuất các biểu đồ PNG chuẩn báo cáo."""
    if "so_anh_da_xet" not in scan_result:
        return {"thong_bao": scan_result.get("thong_bao", "Chưa có kết quả quét để thống kê.")}
    report = Path(scan_result["bao_cao"])
    saved = report / "thong_ke_va_nhom.json"
    if mode == "xem_bao_cao":
        if not saved.is_file():
            return {"thong_bao": "Chưa có bảng thống kê/biểu đồ đã lưu. Chuyển CHE_DO='xu_ly_du_lieu' để tạo."}
        data = json.loads(saved.read_text(encoding="utf-8"))
        if data.get("metadata_zip") != scan_result["metadata_zip"] or data.get("phien_ban_ma") != Y2_VERSION:
            return {"thong_bao": "Báo cáo thống kê không khớp phiên bản/nguồn hiện tại; cần chạy lại xử lý."}
        data["cach_doc"] = "Đọc bảng và biểu đồ đã lưu; không quét nguồn."
        return data

    t0 = time.monotonic()
    rows = y2_load_detail_rows(report)
    group_summary, edges, confirmed_cross = y2_build_groups_and_edges(rows)
    # Lưu lại chi_tiet_anh đã gắn group_id ổn định
    detail_path = report / "chi_tiet_anh.jsonl.gz"
    tmp = detail_path.with_name(f"{detail_path.name}.{uuid.uuid4().hex}.part")
    with gzip.open(tmp, "wt", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    tmp.replace(detail_path)

    scope_tag = "TOÀN BỘ NGUỒN" if scan_result["toan_bo"] else f"MẪU THỬ {scan_result['so_anh_da_xet']:,} ẢNH"
    src_counts = scan_result["so_anh_nguon_theo_tap"]

    # 1. Bảng quy mô theo split (tách rõ đơn vị ảnh và đơn vị khung)
    split_rows = []
    for sp in ["train", "val", "test"]:
        rs = [r for r in rows if r["original_split"] == sp]
        split_rows.append({
            "tap": sp,
            "so_tep_anh_nguon": src_counts.get(sp, 0),
            "so_anh_da_xet": len(rs),
            "anh_doc_duoc": sum(not r["image_error"] for r in rs),
            "anh_loi_doc": sum(bool(r["image_error"]) for r in rs),
            "anh_thieu_hoac_loi_tep_nhan": sum(bool(r["label_file_error"]) for r in rs),
            "anh_nhan_rong": sum((not r["label_file_error"]) and r["box_count_total"] == 0 for r in rs),
            "anh_co_khung_loi": sum(r["box_count_invalid"] > 0 for r in rs),
            "tong_khung_hop_le": sum(r["box_count_valid"] for r in rs),
            "tong_dong_khung_loi": sum(r["box_count_invalid"] for r in rs),
        })
    split_rows.append({
        "tap": "tong",
        **{k: sum(r[k] for r in split_rows) for k in split_rows[0] if k != "tap"},
    })

    # 2. Kích thước ảnh phổ biến
    size_ctr = Counter((r["width_px"], r["height_px"]) for r in rows if not r["image_error"])
    n_readable = sum(size_ctr.values())
    size_rows = [
        {
            "rong_px": w,
            "cao_px": h,
            "kich_thuoc": f"{w}x{h}",
            "so_anh": cnt,
            "ty_le_phan_tram": round(100.0 * cnt / max(1, n_readable), 3),
        }
        for (w, h), cnt in sorted(size_ctr.items(), key=lambda kv: (-kv[1], kv[0]))
    ]

    # 3. Phân bố số khung/ảnh
    box_cnt_ctr = Counter(r["box_count_valid"] for r in rows if not r["image_error"] and not r["label_file_error"])
    n_lbl_readable = sum(box_cnt_ctr.values())
    box_dist_rows = [
        {
            "so_khung_hop_le": k,
            "nhom_mo_ta": "0 (nhãn rỗng)" if k == 0 else ("1 khung" if k == 1 else f"{k} khung"),
            "so_anh": cnt,
            "ty_le_phan_tram": round(100.0 * cnt / max(1, n_lbl_readable), 3),
        }
        for k, cnt in sorted(box_cnt_ctr.items())
    ]

    # 4. Kích thước/tỷ lệ khung ký tự & 5. Tần suất ký tự nguồn
    box_metrics = []
    char_ctr = Counter()
    id_to_char = {c["original_class_id"]: c["character"] for c in scan_result["bang_lop_goc"]}
    for r in rows:
        if r["image_error"]:
            continue
        w, h = r["width_px"], r["height_px"]
        lb_scale = min(640.0 / w, 640.0 / h)
        for b in r["boxes"]:
            if b["bbox_status"] != "hop_le":
                continue
            bw_px = b["raw_yolo_w"] * w
            bh_px = b["raw_yolo_h"] * h
            box_metrics.append({
                "source_image_id": r["source_image_id"],
                "original_split": r["original_split"],
                "original_class_id": b["original_class_id"],
                "character": b["character"],
                "rong_px": round(bw_px, 3),
                "cao_px": round(bh_px, 3),
                "ty_le_rong_cao": round(bw_px / bh_px, 4),
                "ty_le_dien_tich": round(b["raw_yolo_w"] * b["raw_yolo_h"], 6),
                "cao_letterbox_640_px": round(bh_px * lb_scale, 3),
            })
            char_ctr[b["original_class_id"]] += 1

    total_valid_boxes = len(box_metrics)
    char_freq_rows = []
    for c in scan_result["bang_lop_goc"]:
        cid = c["original_class_id"]
        cnt = char_ctr.get(cid, 0)
        char_freq_rows.append({
            "original_class_id": cid,
            "character": c["character"],
            "in_ocr_32": c["in_ocr_32"],
            "so_khung": cnt,
            "ty_le_phan_tram": round(100.0 * cnt / max(1, total_valid_boxes), 4),
        })

    # Tóm tắt phân vị kích thước khung
    box_summary_rows = []
    if box_metrics:
        for col, label in [
            ("rong_px", "Rộng khung (px)"),
            ("cao_px", "Cao khung (px)"),
            ("ty_le_rong_cao", "Tỷ lệ rộng/cao"),
            ("ty_le_dien_tich", "Tỷ lệ diện tích/ảnh"),
            ("cao_letterbox_640_px", "Cao sau letterbox 640 (px)"),
        ]:
            arr = np.array([m[col] for m in box_metrics], dtype=np.float64)
            box_summary_rows.append({
                "chi_so": label,
                "so_khung": len(arr),
                "trung_binh": round(float(arr.mean()), 4),
                "do_lech_chuan": round(float(arr.std()), 4),
                "min": round(float(arr.min()), 4),
                "p10": round(float(np.percentile(arr, 10)), 4),
                "p50": round(float(np.percentile(arr, 50)), 4),
                "p90": round(float(np.percentile(arr, 90)), 4),
                "max": round(float(arr.max()), 4),
            })

    # 6. Bảng lỗi theo SỐ ẢNH DUY NHẤT (có chú thích chồng lặp; tách riêng khỏi số nhóm trùng)
    issue_sets = {
        "Ảnh không đọc được": {r["source_image_id"] for r in rows if r["image_error"]},
        "Ảnh thiếu hoặc lỗi đọc tệp nhãn": {r["source_image_id"] for r in rows if r["label_file_error"]},
        "Ảnh nhãn rỗng (0 khung — cần xác nhận nền)": {r["source_image_id"] for r in rows if not r["label_file_error"] and r["box_count_total"] == 0},
        "Ảnh có ít nhất 1 khung sai cú pháp/vượt biên": {r["source_image_id"] for r in rows if r["box_count_invalid"] > 0},
        "Ảnh có dòng khung lặp trùng hoàn toàn": {r["source_image_id"] for r in rows if r["dup_line_count"] > 0},
        "Ảnh có cặp khung chồng lấn IoU >= 0.85": {r["source_image_id"] for r in rows if r["high_iou_pairs"]},
        "Ảnh có thẻ EXIF xoay khác 1": {r["source_image_id"] for r in rows if r["exif_orientation"] != 1},
        "Ảnh trùng pixel nhưng mâu thuẫn nhãn": {r["source_image_id"] for r in rows if r["pixel_label_conflict"]},
        "Ảnh thuộc nhóm xác nhận đi qua nhiều tập": {r["source_image_id"] for r in rows if r["group_cross_split"]},
    }
    union_issues = set().union(*issue_sets.values()) if issue_sets else set()
    issue_rows = [
        {
            "loai_van_de": k,
            "so_anh_duy_nhat": len(v),
            "ty_le_tren_anh_xet_phan_tram": round(100.0 * len(v) / max(1, len(rows)), 3),
            "ghi_chu": "Các dòng có thể chồng lặp, không cộng dồn trực tiếp",
        }
        for k, v in issue_sets.items()
    ]
    issue_rows.append({
        "loai_van_de": "TỔNG SỐ ẢNH DUY NHẤT CÓ ÍT NHẤT 1 VẤN ĐỀ TRÊN",
        "so_anh_duy_nhat": len(union_issues),
        "ty_le_tren_anh_xet_phan_tram": round(100.0 * len(union_issues) / max(1, len(rows)), 3),
        "ghi_chu": f"Mẫu số: {len(rows):,} ảnh đã xét; tách riêng {scan_result['so_nhan_mo_coi']:,} tệp nhãn mồ côi toàn ZIP",
    })

    # Chất lượng ảnh phụ lục
    qual_rows = [
        {"source_image_id": r["source_image_id"], "original_split": r["original_split"], **r["quality"]}
        for r in rows if r.get("quality")
    ]

    # Lưu tất cả CSV
    y2_write_csv(report / "quy_mo_theo_split.csv", split_rows)
    y2_write_csv(report / "kich_thuoc_anh.csv", size_rows)
    y2_write_csv(report / "phan_bo_so_khung_moi_anh.csv", box_dist_rows)
    y2_write_csv(report / "kich_thuoc_khung_ky_tu.csv", box_metrics)
    y2_write_csv(report / "tom_tat_kich_thuoc_khung.csv", box_summary_rows)
    y2_write_csv(report / "tan_suat_ky_tu_nguon.csv", char_freq_rows)
    y2_write_csv(report / "tom_tat_loi_anh_duy_nhat.csv", issue_rows)
    y2_write_csv(report / "tom_tat_nhom_trung.csv", group_summary)
    y2_write_csv(report / "quan_he_nhom.csv", edges)
    y2_write_csv(report / "chat_luong_anh.csv", qual_rows)

    # --- Vẽ và lưu các biểu đồ PNG (có đơn vị, mẫu số và phạm vi rõ ràng) ---
    charts = {}
    charts_dir = report / "Bieu_do"
    charts_dir.mkdir(exist_ok=True)

    # Hình 1: Quy mô ảnh đọc được và tổng khung hợp lệ theo split (hai trục/ô riêng, không trộn đơn vị)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    sps = ["train", "val", "test"]
    sp_map = {r["tap"]: r for r in split_rows}
    ax1.bar(sps, [sp_map[s]["anh_doc_duoc"] for s in sps], color=["#3b82f6", "#10b981", "#f59e0b"])
    ax1.set(title=f"Số ảnh đọc được theo tập ({scope_tag})", ylabel="Số ảnh")
    ax2.bar(sps, [sp_map[s]["tong_khung_hop_le"] for s in sps], color=["#2563eb", "#059669", "#d97706"])
    ax2.set(title=f"Tổng khung ký tự hợp lệ ({total_valid_boxes:,} khung)", ylabel="Số khung ký tự")
    fig.tight_layout()
    p1 = charts_dir / "01_quy_mo_theo_split.png"
    fig.savefig(p1, dpi=130)
    plt.close(fig)
    charts["quy_mo_split"] = str(p1)

    # Hình 2: Top kích thước ảnh phổ biến + phần còn lại
    top_sz = size_rows[:8]
    other_cnt = sum(r["so_anh"] for r in size_rows[8:])
    labels_sz = [r["kich_thuoc"] for r in top_sz] + (["Kích thước khác"] if other_cnt else [])
    vals_sz = [r["so_anh"] for r in top_sz] + ([other_cnt] if other_cnt else [])
    fig, ax = plt.subplots(figsize=(9.5, 4.0))
    ax.barh(labels_sz[::-1], vals_sz[::-1], color="#4f81bd")
    ax.set(xlabel=f"Số ảnh (mẫu số: {n_readable:,} ảnh đọc được)", title=f"Kích thước ảnh phổ biến — {scope_tag}")
    fig.tight_layout()
    p2 = charts_dir / "02_kich_thuoc_anh.png"
    fig.savefig(p2, dpi=130)
    plt.close(fig)
    charts["kich_thuoc_anh"] = str(p2)

    # Hình 3: Phân bố số khung ký tự / ảnh
    if box_dist_rows:
        fig, ax = plt.subplots(figsize=(9.5, 3.8))
        xs = [str(r["so_khung_hop_le"]) for r in box_dist_rows[:18]]
        ys = [r["so_anh"] for r in box_dist_rows[:18]]
        tail = sum(r["so_anh"] for r in box_dist_rows[18:])
        if tail:
            xs.append(f">= {box_dist_rows[18]['so_khung_hop_le']}")
            ys.append(tail)
        ax.bar(xs, ys, color="#528f6c")
        ax.set(xlabel="Số khung ký tự hợp lệ trên mỗi ảnh", ylabel=f"Số ảnh (mẫu số: {n_lbl_readable:,})",
               title=f"Phân bố số khung ký tự / ảnh — {scope_tag}")
        fig.tight_layout()
        p3 = charts_dir / "03_phan_bo_so_khung_moi_anh.png"
        fig.savefig(p3, dpi=130)
        plt.close(fig)
        charts["phan_bo_so_khung"] = str(p3)

    # Hình 4: Kích thước & tỷ lệ khung ký tự
    if box_metrics:
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
        axes[0].hist([m["rong_px"] for m in box_metrics], bins=30, alpha=0.7, label="Rộng (px)")
        axes[0].hist([m["cao_px"] for m in box_metrics], bins=30, alpha=0.6, label="Cao (px)")
        axes[0].set(xlabel="Pixel", ylabel="Số khung", title="Rộng & cao khung gốc")
        axes[0].legend()
        axes[1].hist([min(m["ty_le_rong_cao"], 4.0) for m in box_metrics], bins=30, color="#d97706")
        axes[1].set(xlabel="Rộng / cao (cắt hiển thị ở 4.0)", ylabel="Số khung", title="Tỷ lệ hình dạng khung")
        axes[2].hist([m["ty_le_dien_tich"] for m in box_metrics], bins=30, color="#7c3aed")
        axes[2].set(xlabel="Diện tích khung / diện tích ảnh", ylabel="Số khung", title=f"Tỷ lệ diện tích ({total_valid_boxes:,} khung)")
        fig.suptitle(f"Đặc điểm kích thước khung ký tự ({scope_tag} — chỉ mô tả, chưa đặt ngưỡng loại)", y=1.02)
        fig.tight_layout()
        p4 = charts_dir / "04_kich_thuoc_khung_ky_tu.png"
        fig.savefig(p4, dpi=130, bbox_inches="tight")
        plt.close(fig)
        charts["kich_thuoc_khung"] = str(p4)

    # Hình 5: Tần suất ký tự nguồn (ID gốc trước khi gộp về lớp 0)
    if char_freq_rows:
        fig, ax = plt.subplots(figsize=(13, 4.2))
        x_lbls = [f"{r['character']} ({r['original_class_id']})" for r in char_freq_rows]
        y_vals = [r["so_khung"] for r in char_freq_rows]
        colors = ["#2563eb" if r["in_ocr_32"] else "#ef4444" for r in char_freq_rows]
        ax.bar(x_lbls, y_vals, color=colors)
        ax.tick_params(axis="x", rotation=60, labelsize=8)
        ax.set(ylabel=f"Số khung (mẫu số: {total_valid_boxes:,})",
               title=f"Tần suất ký tự gốc từ data.yaml — {scope_tag} (Xanh: thuộc 32 lớp OCR; Đỏ: ngoài 32 lớp, vẫn giữ ở YOLO2)")
        fig.tight_layout()
        p5 = charts_dir / "05_tan_suat_ky_tu_nguon.png"
        fig.savefig(p5, dpi=130)
        plt.close(fig)
        charts["tan_suat_ky_tu"] = str(p5)

    res = {
        "phien_ban_ma": Y2_VERSION,
        "thoi_diem": y2_now_iso(),
        "metadata_zip": scan_result["metadata_zip"],
        "pham_vi": scan_result["pham_vi"],
        "toan_bo": scan_result["toan_bo"],
        "so_anh_da_xet": len(rows),
        "tong_khung_hop_le": total_valid_boxes,
        "quy_mo_theo_split": split_rows,
        "kich_thuoc_anh_top": size_rows[:15],
        "tong_so_kich_thuoc_khac_nhau": len(size_rows),
        "phan_bo_so_khung": box_dist_rows,
        "tom_tat_kich_thuoc_khung": box_summary_rows,
        "tan_suat_ky_tu_nguon": char_freq_rows,
        "tom_tat_loi": issue_rows,
        "tom_tat_nhom_trung": group_summary,
        "so_nhom_xac_nhan_qua_nhieu_tap": confirmed_cross,
        "so_canh_quan_he": len(edges),
        "bieu_do": charts,
        "bao_cao": str(report),
        "thoi_gian_giay": round(time.monotonic() - t0, 3),
    }
    y2_write_json(saved, res)
    return res


def y2_step4_build_review_sheets(scan_result, mode="xem_bao_cao", max_cases_per_group=45, local_root="/content/yolo2_tam"):
    """Tạo bảng ảnh đối chiếu Ra_soat/ cho các ca lỗi, nhãn rỗng, trùng pixel mâu thuẫn, qua tập và gợi ý dHash."""
    if "so_anh_da_xet" not in scan_result:
        return {"thong_bao": scan_result.get("thong_bao", "Chưa có kết quả quét để rà soát.")}
    report = Path(scan_result["bao_cao"])
    folder = report / "Ra_soat"
    saved = folder / "ket_qua_ra_soat.json"
    if mode == "xem_bao_cao":
        if not saved.is_file():
            return {"thong_bao": "Chưa có bảng ảnh rà soát đã lưu. Chế độ xem không tự tạo ảnh."}
        data = json.loads(saved.read_text(encoding="utf-8"))
        if data.get("metadata_zip") != scan_result["metadata_zip"] or data.get("phien_ban_ma") != Y2_VERSION:
            return {"thong_bao": "Bảng rà soát đã lưu không khớp phiên bản/nguồn hiện tại."}
        return data

    t0 = time.monotonic()
    rows = y2_load_detail_rows(report)
    id_to_char = {c["original_class_id"]: c["character"] for c in scan_result["bang_lop_goc"]}
    folder.mkdir(parents=True, exist_ok=True)
    zip_src = _y2_resolve_readable_zip(scan_result, local_root)

    # Gom các nhóm cần đối chiếu bằng mắt
    by_px = defaultdict(list)
    by_grp = defaultdict(list)
    by_dh = defaultdict(list)
    for r in rows:
        if r["pixel_sha256"]:
            by_px[r["pixel_sha256"]].append(r)
        by_grp[r["group_id"]].append(r)
        if r["dhash"]:
            by_dh[r["dhash"]].append(r)

    categories = {
        "Trung_pixel_mau_thuan_nhan": [],
        "Trung_pixel_qua_tap": [],
        "Nhom_rf_qua_tap": [],
        "Goi_y_dhash_qua_tap": [],
        "Nhan_loi_can_xem": [],
        "Nhan_rong": [],
        "Khung_chong_lap_iou_cao": [],
    }
    bad_box_numbers = []

    for px, rs in sorted(by_px.items()):
        if len(rs) > 1 and any(r["pixel_label_conflict"] for r in rs):
            for r in sorted(rs, key=lambda x: (-Y2_PRIORITY[x["original_split"]], x["source_member"]))[:2]:
                categories["Trung_pixel_mau_thuan_nhan"].append(dict(r, note=f"Mau thuan nhan | {r['original_split']}"))
        elif len(rs) > 1 and len({r["original_split"] for r in rs}) > 1:
            for r in sorted(rs, key=lambda x: (-Y2_PRIORITY[x["original_split"]], x["source_member"]))[:2]:
                categories["Trung_pixel_qua_tap"].append(dict(r, note=f"Trung pixel qua tap | {r['original_split']}"))

    for gid, rs in sorted(by_grp.items()):
        if len({r["original_split"] for r in rs}) > 1 and not any(len(by_px[r["pixel_sha256"]]) > 1 for r in rs if r["pixel_sha256"]):
            for sp in ["test", "val", "train"]:
                cands = [r for r in rs if r["original_split"] == sp]
                if cands:
                    categories["Nhom_rf_qua_tap"].append(dict(cands[0], note=f"Nhom .rf. qua tap | {sp}"))

    for dh, rs in sorted(by_dh.items()):
        if len({r["original_split"] for r in rs}) > 1 and len({r["group_id"] for r in rs}) > 1:
            reps = {}
            for r in sorted(rs, key=lambda x: (-Y2_PRIORITY[x["original_split"]], x["source_member"])):
                reps.setdefault(r["group_id"], r)
            for r in list(reps.values())[:2]:
                categories["Goi_y_dhash_qua_tap"].append(dict(r, note=f"Goi y dHash {dh[:8]} | {r['original_split']}"))

    for r in rows:
        if r["box_count_invalid"] > 0:
            categories["Nhan_loi_can_xem"].append(dict(r, note=f"Khung loi: {r['box_count_invalid']}/{r['box_count_total']}"))
            for b in r["boxes"]:
                if b["bbox_status"] == "khung_loi":
                    bad_box_numbers.append({
                        "source_image_id": r["source_image_id"],
                        "source_box_id": b["source_box_id"],
                        "source_line_index": b["source_line_index"],
                        "original_split": r["original_split"],
                        "original_class_id": b["original_class_id"],
                        "raw_yolo_cx": b["raw_yolo_cx"],
                        "raw_yolo_cy": b["raw_yolo_cy"],
                        "raw_yolo_w": b["raw_yolo_w"],
                        "raw_yolo_h": b["raw_yolo_h"],
                        "overshoot_px": b["overshoot_px"],
                        "reason_codes": ";".join(b["reason_codes"]),
                    })
        if not r["label_file_error"] and r["box_count_total"] == 0 and not r["image_error"]:
            categories["Nhan_rong"].append(dict(r, note="Nhan rong: kiem tra co ky tu hay khong"))
        if r["high_iou_pairs"]:
            max_iou = max(p["iou"] for p in r["high_iou_pairs"])
            categories["Khung_chong_lap_iou_cao"].append(dict(r, note=f"{len(r['high_iou_pairs'])} cap IoU max={max_iou:.2f}"))

    y2_write_csv(folder / "tri_so_khung_loi.csv", bad_box_numbers)
    pages = {}
    index_rows = []
    for cat, items in categories.items():
        subset = items[:max_cases_per_group]
        cat_pages = []
        for p_idx in range(0, len(subset), 9):
            chunk = subset[p_idx : p_idx + 9]
            out_img = folder / cat / f"Trang_{p_idx // 9 + 1:02d}.jpg"
            y2_render_labeled_grid(zip_src, chunk, id_to_char, out_img, cols=3)
            cat_pages.append(str(out_img))
            for item in chunk:
                index_rows.append({
                    "loai": cat,
                    "trang": p_idx // 9 + 1,
                    "source_image_id": item["source_image_id"],
                    "original_split": item["original_split"],
                    "group_id": item["group_id"],
                    "note": item.get("note", ""),
                })
        pages[cat] = cat_pages
    y2_write_csv(folder / "anh_can_doi_chieu.csv", index_rows)

    # Tạo tệp mẫu quyet_dinh_ra_soat.json nếu chưa tồn tại (không ghi đè quyết định đã lưu)
    dec_path = folder / "quyet_dinh_ra_soat.json"
    if not dec_path.is_file():
        y2_write_json(
            dec_path,
            {
                "phien_ban_quyet_dinh": "1.0",
                "metadata_zip": scan_result["metadata_zip"],
                "ghi_chu": "Điền quyết định sau khi xem bảng ảnh Ra_soat; mặc định không tự sửa hay tự nối dHash.",
                "xac_nhan_anh_nen_khong_ky_tu": [],
                "sua_khung_vuot_bien_theo_box_id": [],
                "gioi_han_vuot_bien_toi_da_px": 2.0,
                "noi_them_nhom_dhash_xac_nhan": [],
                "giai_quyet_mau_thuan_pixel_giu_image_id": [],
            },
        )

    res = {
        "phien_ban_ma": Y2_VERSION,
        "metadata_zip": scan_result["metadata_zip"],
        "thu_muc": str(folder),
        "so_luong_ca": {k: len(v) for k, v in categories.items()},
        "so_khung_loi_ghi_nhan": len(bad_box_numbers),
        "trang_anh": pages,
        "tep_quyet_dinh": str(dec_path),
        "thoi_gian_giay": round(time.monotonic() - t0, 3),
    }
    y2_write_json(saved, res)
    return res


In [ ]:
# @title Hàm hỗ trợ Phần 3: phương án xử lý, ảnh đại diện, xuất YOLO2, bàn giao OCR và kiểm chứng — có thể thu gọn
"""Áp dụng quy tắc giữ/loại, xuất bộ yolo2_char_num (lớp 0), gói Ban_giao_OCR (nhãn gốc) và kiểm chứng độc lập."""
import tempfile


def _y2_clip_box_verified(box, width_px, height_px, max_overshoot_px=2.0):
    """Chỉ sửa khung vượt biên đã được xác nhận bằng mắt; lưu trị số trước–sau và kiểm tra lại."""
    if box["reason_codes"] != ["vuot_bien_anh"] or box["overshoot_px"] > max_overshoot_px:
        raise ValueError(f"Khung {box['source_box_id']} vượt ngoài phạm vi sửa an toàn ({box['overshoot_px']} px).")
    xn1 = max(0.0, min(1.0, box["raw_yolo_cx"] - box["raw_yolo_w"] / 2.0))
    yn1 = max(0.0, min(1.0, box["raw_yolo_cy"] - box["raw_yolo_h"] / 2.0))
    xn2 = max(0.0, min(1.0, box["raw_yolo_cx"] + box["raw_yolo_w"] / 2.0))
    yn2 = max(0.0, min(1.0, box["raw_yolo_cy"] + box["raw_yolo_h"] / 2.0))
    bw, bh = xn2 - xn1, yn2 - yn1
    if bw <= 0.0 or bh <= 0.0:
        raise ValueError(f"Khung {box['source_box_id']} không còn diện tích dương sau chỉnh biên.")
    cx, cy = (xn1 + xn2) / 2.0, (yn1 + yn2) / 2.0
    fixed = dict(box)
    fixed.update({
        "raw_yolo_cx": cx,
        "raw_yolo_cy": cy,
        "raw_yolo_w": bw,
        "raw_yolo_h": bh,
        "x1_px": round(xn1 * width_px, 4),
        "y1_px": round(yn1 * height_px, 4),
        "x2_px": round(xn2 * width_px, 4),
        "y2_px": round(yn2 * height_px, 4),
        "overshoot_px": 0.0,
        "bbox_status": "hop_le_sau_sua_bien",
        "reason_codes": ["da_chinh_vao_bien_co_xac_nhan"],
    })
    return fixed


def y2_step5_make_plan(scan_result, mode="xem_bao_cao", single_rep_val_test=True):
    """Lập phương án giữ/tạm giữ ngoài theo quy tắc test -> val -> train; kiểm chứng 0 nhóm xác nhận qua tập."""
    if "so_anh_da_xet" not in scan_result:
        return {"thong_bao": scan_result.get("thong_bao", "Chưa có kết quả quét để lập phương án.")}
    report = Path(scan_result["bao_cao"])
    saved = report / "phuong_an_xu_ly.json"
    if mode == "xem_bao_cao":
        if not saved.is_file():
            return {"thong_bao": "Chưa có phương án xử lý đã lưu. Chuyển CHE_DO='xu_ly_du_lieu' để lập phương án."}
        data = json.loads(saved.read_text(encoding="utf-8"))
        if data.get("metadata_zip") != scan_result["metadata_zip"] or data.get("phien_ban_ma") != Y2_VERSION:
            return {"thong_bao": "Phương án đã lưu không khớp phiên bản/nguồn hiện tại."}
        return data

    t0 = time.monotonic()
    rows = y2_load_detail_rows(report)
    dec_path = report / "Ra_soat/quyet_dinh_ra_soat.json"
    decisions = {}
    if dec_path.is_file():
        loaded = json.loads(dec_path.read_text(encoding="utf-8"))
        if loaded.get("metadata_zip") == scan_result["metadata_zip"]:
            decisions = loaded

    confirmed_bg = set(decisions.get("xac_nhan_anh_nen_khong_ky_tu", []))
    repair_box_ids = set(decisions.get("sua_khung_vuot_bien_theo_box_id", []))
    max_overshoot = float(decisions.get("gioi_han_vuot_bien_toi_da_px", 2.0))
    resolved_conflict_keep = set(decisions.get("giai_quyet_mau_thuan_pixel_giu_image_id", []))
    manual_dhash_merges = decisions.get("noi_them_nhom_dhash_xac_nhan", [])

    # Hợp nhất thêm các nhóm dHash CHỈ KHI người dùng đã ghi xác nhận trong quyet_dinh_ra_soat.json
    g_parent = {r["group_id"]: r["group_id"] for r in rows}

    def g_find(g):
        while g_parent[g] != g:
            g_parent[g] = g_parent[g_parent[g]]
            g = g_parent[g]
        return g

    def g_union(a, b):
        ra, rb = g_find(a), g_find(b)
        if ra != rb:
            g_parent[rb] = ra

    id_to_gid = {r["source_image_id"]: r["group_id"] for r in rows}
    for pair in manual_dhash_merges:
        if len(pair) >= 2 and all(img_id in id_to_gid for img_id in pair):
            base_g = id_to_gid[pair[0]]
            for img_id in pair[1:]:
                g_union(base_g, id_to_gid[img_id])

    grouped_rows = defaultdict(list)
    for r in rows:
        r["final_group_id"] = g_find(r["group_id"])
        grouped_rows[r["final_group_id"]].append(r)

    # Ưu tiên test (2) -> val (1) -> train (0) cho toàn bộ họ biến thể trong nhóm đã xác nhận
    group_top_prio = {
        g: max(Y2_PRIORITY[r["original_split"]] for r in rs)
        for g, rs in grouped_rows.items()
    }

    repair_log = []
    # Xử lý từng ảnh: kiểm tra chất lượng cơ bản + sửa khung có xác nhận
    base_flags = {}
    effective_boxes_by_img = {}
    for r in rows:
        flags = []
        if r["image_error"]:
            flags.append("anh_khong_doc_duoc")
        if r["label_file_error"]:
            flags.append(r["label_file_error"])
        eff_boxes = []
        unresolved_bad = 0
        for b in r["boxes"]:
            if b["bbox_status"] == "hop_le":
                eff_boxes.append(b)
            elif b["bbox_status"] == "khung_lap_trung":
                # Dòng lặp hoàn toàn giống hệt đã được giữ 1 bản ở hop_le
                continue
            elif b["source_box_id"] in repair_box_ids and not r["image_error"]:
                fixed = _y2_clip_box_verified(b, r["width_px"], r["height_px"], max_overshoot)
                eff_boxes.append(fixed)
                repair_log.append({
                    "source_image_id": r["source_image_id"],
                    "source_box_id": b["source_box_id"],
                    "before_cx": b["raw_yolo_cx"], "before_cy": b["raw_yolo_cy"],
                    "before_w": b["raw_yolo_w"], "before_h": b["raw_yolo_h"],
                    "after_cx": fixed["raw_yolo_cx"], "after_cy": fixed["raw_yolo_cy"],
                    "after_w": fixed["raw_yolo_w"], "after_h": fixed["raw_yolo_h"],
                    "overshoot_px_before": b["overshoot_px"],
                })
            else:
                unresolved_bad += 1
        if unresolved_bad > 0:
            flags.append("tat_ca_khung_loi" if not eff_boxes else "co_mot_phan_khung_loi_can_xem")
        if not r["label_file_error"] and r["box_count_total"] == 0 and r["source_image_id"] not in confirmed_bg:
            flags.append("nhan_rong_can_xac_nhan_anh_nen")
        if r["pixel_label_conflict"] and r["source_image_id"] not in resolved_conflict_keep:
            flags.append("trung_pixel_mau_thuan_nhan")
        base_flags[r["source_image_id"]] = flags
        effective_boxes_by_img[r["source_image_id"]] = eff_boxes

    # Chọn đại diện khi trùng pixel cùng tập (hoặc đánh giá val/test theo nhóm)
    chosen_eval_by_group = {}
    if single_rep_val_test:
        for g, rs in grouped_rows.items():
            if group_top_prio[g] > 0:
                cands = [
                    r for r in rs
                    if Y2_PRIORITY[r["original_split"]] == group_top_prio[g]
                    and not base_flags[r["source_image_id"]]
                ]
                if cands:
                    cands.sort(key=lambda r: (-len(effective_boxes_by_img[r["source_image_id"]]), -(r["width_px"] * r["height_px"]), r["source_image_id"]))
                    chosen_eval_by_group[g] = cands[0]["source_image_id"]

    # Trong cùng tập train, nếu có ảnh giống HỆT pixel (nhãn giống nhau), chỉ giữ 1 bản đại diện
    chosen_pixel_in_split = {}
    by_split_px = defaultdict(list)
    for r in rows:
        if r["pixel_sha256"] and not base_flags[r["source_image_id"]] and Y2_PRIORITY[r["original_split"]] == group_top_prio[r["final_group_id"]]:
            by_split_px[(r["original_split"], r["pixel_sha256"])].append(r)
    for key, cands in by_split_px.items():
        cands.sort(key=lambda r: (-len(effective_boxes_by_img[r["source_image_id"]]), r["source_image_id"]))
        chosen_pixel_in_split[key] = cands[0]["source_image_id"]

    plan_rows = []
    for r in rows:
        sid = r["source_image_id"]
        g = r["final_group_id"]
        sp = r["original_split"]
        flags = list(base_flags[sid])
        if Y2_PRIORITY[sp] < group_top_prio[g]:
            flags.append("nhom_xac_nhan_o_tap_uu_tien_cao_hon")
        elif single_rep_val_test and group_top_prio[g] > 0 and g in chosen_eval_by_group and sid != chosen_eval_by_group[g]:
            flags.append("chi_giu_mot_dai_dien_nhom_val_test")
        elif r["pixel_sha256"] and (sp, r["pixel_sha256"]) in chosen_pixel_in_split and sid != chosen_pixel_in_split[(sp, r["pixel_sha256"])]:
            flags.append("trung_pixel_cung_tap_giu_dai_dien")

        if "anh_khong_doc_duoc" in flags:
            status = "loai"
        elif flags:
            status = "tam_giu_ngoai"
        else:
            status = "giu"

        eff_boxes = effective_boxes_by_img[sid]
        plan_rows.append({
            "source_image_id": sid,
            "source_member": r["source_member"],
            "original_split": sp,
            "final_split": sp if status == "giu" else "",
            "parent_id": r["parent_id"],
            "group_id": g,
            "status": status,
            "reason_codes": ";".join(flags),
            "box_count_before": r["box_count_valid"],
            "box_count_after": len(eff_boxes) if status == "giu" else 0,
            "repaired_boxes": sum(1 for b in eff_boxes if b["bbox_status"] == "hop_le_sau_sua_bien"),
        })

    # Kiểm chứng bắt buộc: KHÔNG còn nhóm đã xác nhận nào xuất hiện ở >= 2 tập trong bộ giữ lại
    kept_splits_by_group = defaultdict(set)
    for p in plan_rows:
        if p["status"] == "giu":
            kept_splits_by_group[p["group_id"]].add(p["final_split"])
    leaked = [g for g, sps in kept_splits_by_group.items() if len(sps) > 1]
    if leaked:
        raise AssertionError(f"Lỗi quy tắc: vẫn còn {len(leaked)} nhóm đã xác nhận nằm ở nhiều tập!")

    # Bảng trước–sau theo split (đếm ảnh duy nhất và tổng khung)
    before_after = []
    for sp in ["train", "val", "test"]:
        ps = [p for p in plan_rows if p["original_split"] == sp]
        kept_ps = [p for p in ps if p["status"] == "giu"]
        before_after.append({
            "tap": sp,
            "anh_ban_dau": len(ps),
            "anh_du_kien_giu": len(kept_ps),
            "anh_tam_giu_ngoai_hoac_loai": len(ps) - len(kept_ps),
            "khung_hop_le_ban_dau": sum(p["box_count_before"] for p in ps),
            "khung_du_kien_giu": sum(p["box_count_after"] for p in kept_ps),
            "trang_thai_so_lieu": "phuong_an_du_kien_chua_xuat",
        })
    before_after.append({
        "tap": "tong",
        "anh_ban_dau": sum(r["anh_ban_dau"] for r in before_after),
        "anh_du_kien_giu": sum(r["anh_du_kien_giu"] for r in before_after),
        "anh_tam_giu_ngoai_hoac_loai": sum(r["anh_tam_giu_ngoai_hoac_loai"] for r in before_after),
        "khung_hop_le_ban_dau": sum(r["khung_hop_le_ban_dau"] for r in before_after),
        "khung_du_kien_giu": sum(r["khung_du_kien_giu"] for r in before_after),
        "trang_thai_so_lieu": "phuong_an_du_kien_chua_xuat",
    })

    reason_ctr = Counter()
    for p in plan_rows:
        if p["status"] != "giu":
            for code in p["reason_codes"].split(";"):
                if code:
                    reason_ctr[code] += 1
    reason_rows = [
        {"ma_ly_do": k, "so_anh_duy_nhat": v, "ghi_chu": "Một ảnh có thể mang nhiều lý do"}
        for k, v in reason_ctr.most_common()
    ]

    y2_write_csv(report / "phuong_an_tung_anh.csv", plan_rows)
    y2_write_csv(report / "truoc_sau_xu_ly.csv", before_after)
    y2_write_csv(report / "ly_do_tam_giu_ngoai.csv", reason_rows)
    if repair_log:
        y2_write_csv(report / "lich_su_sua_khung.csv", repair_log)

    # Biểu đồ trước–sau xử lý
    charts_dir = report / "Bieu_do"
    charts_dir.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(8.5, 4.0))
    sps = ["train", "val", "test"]
    ba_map = {r["tap"]: r for r in before_after}
    x = np.arange(len(sps))
    ax.bar(x - 0.18, [ba_map[s]["anh_ban_dau"] for s in sps], 0.34, label="Ảnh ban đầu (đã xét)", color="#94a3b8")
    ax.bar(x + 0.18, [ba_map[s]["anh_du_kien_giu"] for s in sps], 0.34, label="Dự kiến giữ (phương án)", color="#10b981")
    ax.set_xticks(x)
    ax.set_xticklabels(sps)
    scope_tag = "TOÀN BỘ NGUỒN" if scan_result["toan_bo"] else f"MẪU THỬ {scan_result['so_anh_da_xet']:,} ẢNH"
    ax.set(ylabel="Số ảnh duy nhất", title=f"So sánh trước–sau xử lý (PHƯƠNG ÁN DỰ KIẾN — {scope_tag})")
    ax.legend()
    fig.tight_layout()
    chart_ba = charts_dir / "06_truoc_sau_xu_ly.png"
    fig.savefig(chart_ba, dpi=130)
    plt.close(fig)

    res = {
        "phien_ban_ma": Y2_VERSION,
        "thoi_diem": y2_now_iso(),
        "metadata_zip": scan_result["metadata_zip"],
        "pham_vi": scan_result["pham_vi"],
        "toan_bo": scan_result["toan_bo"],
        "chi_mot_dai_dien_val_test": single_rep_val_test,
        "truoc_sau": before_after,
        "ly_do_giu_ngoai": reason_rows,
        "so_nhom_xac_nhan_qua_tap_trong_bo_giu": 0,
        "so_khung_da_chinh_bien": len(repair_log),
        "bieu_do_truoc_sau": str(chart_ba),
        "bao_cao": str(report),
        "luu_y_gioi_han": (
            "Đã kiểm tra không còn nhóm ĐÃ XÁC NHẬN (pixel giống nhau, họ tên .rf., hoặc dHash đã xác nhận bằng mắt) "
            "xuất hiện đồng thời ở nhiều tập. Điều này chưa bảo đảm mọi biển số hoặc xe ngoài phạm vi kiểm tra đều độc lập."
        ),
        "thoi_gian_giay": round(time.monotonic() - t0, 3),
    }
    y2_write_json(saved, res)
    return res


def y2_step6_representatives(
    project,
    scan_result,
    plan_result,
    mode="xem_bao_cao",
    n_sizes=5,
    per_size=3,
    rep_scope="bo_giu_lai",
    local_root="/content/yolo2_tam",
):
    """Xuất ảnh đại diện nguyên byte gốc vào 04_Anh_dai_dien/YOLO2/Roboflow/<Ban_pham_vi>/<rong>x<cao>/."""
    if "so_anh_da_xet" not in scan_result:
        return {"thong_bao": scan_result.get("thong_bao", "Chưa có kết quả quét.")}
    if rep_scope not in {"bo_giu_lai", "nguon_ban_dau"}:
        raise ValueError("PHAM_VI_ANH_DAI_DIEN phải là 'bo_giu_lai' hoặc 'nguon_ban_dau'.")
    report = Path(scan_result["bao_cao"])
    saved = report / f"anh_dai_dien_{rep_scope}.json"
    if mode == "xem_bao_cao":
        if not saved.is_file():
            return {"thong_bao": f"Chưa có ảnh đại diện đã lưu ({rep_scope}). Chế độ xem không tự trích ảnh."}
        return json.loads(saved.read_text(encoding="utf-8"))

    rows = y2_load_detail_rows(report)
    kept_ids = None
    if rep_scope == "bo_giu_lai":
        plan_csv = report / "phuong_an_tung_anh.csv"
        if not plan_csv.is_file():
            raise ValueError("Cần chạy Mục 5 (lập phương án) trước khi chọn ảnh đại diện từ bộ giữ lại.")
        kept_ids = {r["source_image_id"] for r in y2_read_csv(plan_csv) if r["status"] == "giu"}

    pool = [
        r for r in rows
        if not r["image_error"] and (kept_ids is None or r["source_image_id"] in kept_ids)
    ]
    size_ctr = Counter((r["width_px"], r["height_px"]) for r in pool)
    top_sizes = sorted(size_ctr.items(), key=lambda kv: (-kv[1], kv[0]))[:n_sizes]

    folder_tag = f"{scan_result['pham_vi']}_{rep_scope}"
    out_root = Path(project) / "04_Anh_dai_dien/YOLO2/Roboflow" / folder_tag
    out_root.mkdir(parents=True, exist_ok=True)
    zip_src = _y2_resolve_readable_zip(scan_result, local_root)

    manifest = []
    chosen_rows = []
    with ZipFile(zip_src) as z:
        for (w, h), cnt in top_sizes:
            cands = [r for r in pool if r["width_px"] == w and r["height_px"] == h]
            cands.sort(key=lambda r: hashlib.sha256(f"{scan_result['seed']}::{r['source_image_id']}".encode()).hexdigest())
            chosen, used_px, used_grp = [], set(), set()
            for r in cands:
                if r["pixel_sha256"] in used_px or r["group_id"] in used_grp:
                    continue
                chosen.append(r)
                used_px.add(r["pixel_sha256"])
                used_grp.add(r["group_id"])
                if len(chosen) >= per_size:
                    break
            sz_dir = out_root / f"{w}x{h}"
            sz_dir.mkdir(parents=True, exist_ok=True)
            for idx_i, r in enumerate(chosen, 1):
                ext = PurePosixPath(r["source_member"]).suffix.lower()
                dest = sz_dir / f"{idx_i:02d}_{r['original_split']}{ext}"
                raw_bytes = z.read(r["source_member"])
                dest.write_bytes(raw_bytes)
                chosen_rows.append(r)
                manifest.append({
                    "source": Y2_SOURCE_NAME,
                    "pham_vi_chon": rep_scope,
                    "pham_vi_du_lieu": scan_result["pham_vi"],
                    "kich_thuoc": f"{w}x{h}",
                    "so_anh_trong_nhom_kich_thuoc": cnt,
                    "original_split": r["original_split"],
                    "group_id": r["group_id"],
                    "source_image_id": r["source_image_id"],
                    "source_member": r["source_member"],
                    "tep_mau_tuong_doi": str(dest.relative_to(out_root).as_posix()),
                    "sha256_byte_goc": hashlib.sha256(raw_bytes).hexdigest(),
                })
    y2_write_csv(out_root / "danh_sach_anh_mau.csv", manifest)
    y2_write_text_atomic(
        out_root / "Giai_thich.txt",
        f"Ảnh đại diện YOLO2 nguồn Roboflow — phạm vi dữ liệu: {scan_result['pham_vi']} — tập chọn: {rep_scope}.\n"
        f"Tối đa {n_sizes} kích thước phổ biến x {per_size} ảnh/kích thước (không ép hạn ngạch nếu thiếu ảnh).\n"
        "Ảnh nộp giữ nguyên byte gốc từ archive, không vẽ khung. Hình vẽ khung nằm riêng trong thư mục báo cáo.\n",
    )
    id_to_char = {c["original_class_id"]: c["character"] for c in scan_result["bang_lop_goc"]}
    preview_path = y2_render_labeled_grid(
        zip_src, chosen_rows, id_to_char, report / f"anh_dai_dien_co_khung_{rep_scope}.png", cols=3
    )
    res = {
        "phien_ban_ma": Y2_VERSION,
        "pham_vi_chon": rep_scope,
        "thu_muc_anh_nop": str(out_root),
        "hinh_ve_khung_bao_cao": preview_path,
        "so_anh_da_xuat": len(manifest),
        "danh_sach": manifest,
    }
    y2_write_json(saved, res)
    return res


def y2_step6_export_and_verify(
    project,
    scan_result,
    plan_result,
    phien_ban_xuat="Ban_01",
    mode="xem_bao_cao",
    class_mapping_confirmed=False,
    do_export=False,
    make_zip=True,
    allow_trial_export=False,
    overwrite=False,
    local_root="/content/yolo2_tam",
):
    """Mốc D & E: xuất yolo2_char_num (lớp 0), gói Ban_giao_OCR (nhãn gốc) và kiểm chứng độc lập sau giải nén."""
    if "so_anh_da_xet" not in scan_result:
        return {"thong_bao": scan_result.get("thong_bao", "Chưa có kết quả quét.")}
    report = Path(scan_result["bao_cao"])
    verify_saved = report / "ket_qua_xuat_va_kiem_chung.json"
    if mode == "xem_bao_cao" or not do_export:
        if verify_saved.is_file():
            data = json.loads(verify_saved.read_text(encoding="utf-8"))
            data["cach_doc"] = "Đọc kết quả xuất & kiểm chứng đã lưu; không ghi đè dữ liệu."
            return data
        return {
            "thong_bao": (
                "Chưa có bộ xuất cho phạm vi này. Sau khi xác nhận bảng lớp (Mốc A) và kiểm tra phương án (Mục 5), "
                "đặt CHE_DO='xu_ly_du_lieu', XAC_NHAN_BANG_LOP_GOC=True và XUAT_BO_DU_LIEU=True."
            )
        }

    if not class_mapping_confirmed:
        raise ValueError(
            "Chưa xác nhận bảng lớp gốc (XAC_NHAN_BANG_LOP_GOC=False). "
            "Cần xem lưới ảnh thật Mốc A để xác nhận data.yaml đúng là khung ký tự trước khi chuyển detector sang lớp 0."
        )
    if not scan_result["toan_bo"] and not allow_trial_export:
        raise ValueError(
            "Đang ở phạm vi MẪU THỬ (SO_ANH_THU > 0). Đặt SO_ANH_THU=0 để xuất bản đầy đủ, "
            "hoặc bật CHO_PHEP_XUAT_MAU_THU=True để xuất bản mẫu có giới hạn."
        )

    t0 = time.monotonic()
    export_tag = phien_ban_xuat if scan_result["toan_bo"] else scan_result["pham_vi"]
    out_base = Path(project) / "02_Du_lieu_huan_luyen/YOLO2" / export_tag
    dataset_dir = out_base / "yolo2_char_num"
    if (dataset_dir / "data.yaml").is_file() and not overwrite:
        raise FileExistsError(
            f"Bản xuất {dataset_dir} đã tồn tại. Không tự ghi đè bản đã xuất; "
            "hãy đổi PHIEN_BAN_XUAT (ví dụ Ban_02) hoặc bật GHI_DE_BAN_XUAT=True."
        )
    if dataset_dir.exists() and overwrite:
        shutil.rmtree(dataset_dir)

    rows = y2_load_detail_rows(report)
    plan_by_id = {r["source_image_id"]: r for r in y2_read_csv(report / "phuong_an_tung_anh.csv")}
    edges = y2_read_csv(report / "quan_he_nhom.csv") if (report / "quan_he_nhom.csv").is_file() else []
    repairs = y2_read_csv(report / "lich_su_sua_khung.csv") if (report / "lich_su_sua_khung.csv").is_file() else []
    repair_by_box = {r["source_box_id"]: r for r in repairs}

    zip_src = _y2_resolve_readable_zip(scan_result, local_root)

    for sp in ["train", "val", "test"]:
        (dataset_dir / "images" / sp).mkdir(parents=True, exist_ok=True)
        (dataset_dir / "labels" / sp).mkdir(parents=True, exist_ok=True)

    # 1. Xuất bộ huấn luyện YOLO2 (lớp 0: char_num)
    manifest_rows = []
    exported_counts = Counter()
    exported_box_counts = Counter()
    used_stems = {sp: set() for sp in ["train", "val", "test"]}

    with ZipFile(zip_src) as z:
        for r in rows:
            sid = r["source_image_id"]
            p = plan_by_id[sid]
            if p["status"] != "giu":
                continue
            sp = p["final_split"]
            src_p = PurePosixPath(r["source_member"])
            ext = src_p.suffix.lower()
            safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", src_p.stem).strip("._") or "img"
            stem = safe_stem[:100]
            if stem in used_stems[sp]:
                stem = f"{stem}_{hashlib.sha1(sid.encode()).hexdigest()[:8]}"
            used_stems[sp].add(stem)

            img_rel = f"images/{sp}/{stem}{ext}"
            lbl_rel = f"labels/{sp}/{stem}.txt"
            raw_img = z.read(r["source_member"])
            (dataset_dir / img_rel).write_bytes(raw_img)

            # Viết nhãn YOLO lớp 0 (char_num), không lọc ký tự ngoài 32 lớp OCR
            lbl_lines = []
            repaired_here = 0
            for b in r["boxes"]:
                if b["bbox_status"] == "hop_le":
                    cx, cy, bw, bh = b["raw_yolo_cx"], b["raw_yolo_cy"], b["raw_yolo_w"], b["raw_yolo_h"]
                    lbl_lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                elif b["source_box_id"] in repair_by_box:
                    rep = repair_by_box[b["source_box_id"]]
                    lbl_lines.append(
                        f"0 {float(rep['after_cx']):.6f} {float(rep['after_cy']):.6f} "
                        f"{float(rep['after_w']):.6f} {float(rep['after_h']):.6f}"
                    )
                    repaired_here += 1
            lbl_text = ("\n".join(lbl_lines) + "\n") if lbl_lines else ""
            (dataset_dir / lbl_rel).write_text(lbl_text, encoding="utf-8")

            exported_counts[sp] += 1
            exported_box_counts[sp] += len(lbl_lines)
            manifest_rows.append({
                "output_image": img_rel,
                "output_label": lbl_rel,
                "source": Y2_SOURCE_NAME,
                "source_file": scan_result["ten_tep_nguon"],
                "source_member": r["source_member"],
                "source_image_id": sid,
                "original_split": r["original_split"],
                "final_split": sp,
                "parent_id": p["parent_id"],
                "group_id": p["group_id"],
                "status": "giu",
                "box_count": len(lbl_lines),
                "repaired_boxes": repaired_here,
                "sha256_output_image": hashlib.sha256(raw_img).hexdigest(),
                "sha256_output_label": hashlib.sha256(lbl_text.encode("utf-8")).hexdigest(),
            })

    y2_write_csv(dataset_dir / "manifest.csv", manifest_rows)
    data_yaml_text = (
        "path: .\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        "nc: 1\n"
        "names:\n"
        "  0: char_num\n"
    )
    y2_write_text_atomic(dataset_dir / "data.yaml", data_yaml_text)

    export_scope_status = "da_xuat_day_du" if scan_result["toan_bo"] else "ban_mau_co_gioi_han"
    readme_yolo2 = (
        f"# Bộ huấn luyện YOLO2 — `{export_tag}` (`yolo2_char_num`)\n\n"
        f"- **Trạng thái:** `{export_scope_status}` ({y2_now_iso()})\n"
        f"- **Vai trò:** Phát hiện vị trí ký tự gồm **1 lớp duy nhất `0: char_num`**. Không lọc khung theo phạm vi 32 lớp OCR.\n"
        f"- **Nguồn:** `{scan_result['ten_tep_nguon']}` (`{Y2_EXPECTED_PROJECT}`, v{Y2_EXPECTED_VERSION}; `metadata_zip={scan_result['metadata_zip']}`).\n"
        f"- **Số lượng đã xuất:**\n"
        f"  - `train`: {exported_counts['train']:,} ảnh, {exported_box_counts['train']:,} khung\n"
        f"  - `val`: {exported_counts['val']:,} ảnh, {exported_box_counts['val']:,} khung\n"
        f"  - `test`: {exported_counts['test']:,} ảnh, {exported_box_counts['test']:,} khung\n"
        f"  - **Tổng:** {sum(exported_counts.values()):,} ảnh, {sum(exported_box_counts.values()):,} khung\n"
        f"- **Cấu trúc nạp:** `data.yaml` dùng `path: .`, trỏ tới `images/train`, `images/val`, `images/test` và tự ghép `labels/<split>` cùng stem.\n"
        f"- **Giới hạn kiểm tra:** Không còn nhóm đã xác nhận (pixel giống nhau, họ tên `.rf.`, hoặc dHash đã xác nhận bằng mắt) xuất hiện ở nhiều tập; điều này chưa chứng minh mọi biển/xe ngoài phạm vi kiểm tra đã độc lập.\n"
    )
    y2_write_text_atomic(dataset_dir / "README.md", readme_yolo2)

    zip_out_path = ""
    if make_zip:
        z_target = out_base / "yolo2_char_num.zip"
        z_stage = (Path(local_root) if local_root else out_base)
        z_stage.mkdir(parents=True, exist_ok=True)
        z_local_tmp = z_stage / f"yolo2_char_num.{uuid.uuid4().hex}.zip.part"
        with ZipFile(z_local_tmp, "w") as zout:
            # Ghi rõ thư mục chuẩn để giải nén ở vị trí khác luôn có đủ train/val/test kể cả khi tập rỗng
            for kind in ("images", "labels"):
                for sp in ("train", "val", "test"):
                    zout.writestr(f"yolo2_char_num/{kind}/{sp}/", "")
            for fp in sorted(dataset_dir.rglob("*")):
                if fp.is_file():
                    zout.write(fp, arcname=str(fp.relative_to(out_base).as_posix()))
        z_drive_tmp = z_target.with_name(f"{z_target.name}.{uuid.uuid4().hex}.part")
        shutil.copyfile(z_local_tmp, z_drive_tmp)
        z_local_tmp.unlink(missing_ok=True)
        z_drive_tmp.replace(z_target)
        zip_out_path = str(z_target)

    # 2. Bắt buộc bàn giao riêng cho OCR (Ban_giao_OCR/ — giữ nguyên nhãn gốc, bao gồm mọi ảnh nguồn)
    ocr_dir = report / "Ban_giao_OCR"
    ocr_dir.mkdir(parents=True, exist_ok=True)
    y2_write_json(
        ocr_dir / "bang_lop_goc.json",
        {
            "schema_version": Y2_SCHEMA_VERSION,
            "source": Y2_SOURCE_NAME,
            "source_file": scan_result["ten_tep_nguon"],
            "source_identity": scan_result["danh_tinh"],
            "verification_status": "da_xac_nhan_tren_anh_moc_a",
            "ocr_32_charset": Y2_OCR_32_CHARS,
            "nc": len(scan_result["bang_lop_goc"]),
            "classes": scan_result["bang_lop_goc"],
        },
    )

    anh_bien_fields = [
        "source", "source_file", "source_member", "source_image_id", "original_split",
        "parent_id", "group_id", "width_px", "height_px", "pixel_sha256", "status", "reason_codes",
        "yolo2_final_split", "exif_orientation", "label_member", "box_count_total", "box_count_valid", "dhash",
    ]
    ky_tu_fields = [
        "source", "source_file", "source_member", "source_image_id", "original_split",
        "parent_id", "group_id", "width_px", "height_px", "pixel_sha256", "status", "reason_codes",
        "source_box_id", "source_line_index", "original_class_id", "character",
        "x1_px", "y1_px", "x2_px", "y2_px", "bbox_status",
        "raw_yolo_cx", "raw_yolo_cy", "raw_yolo_w", "raw_yolo_h", "in_ocr_32",
    ]
    anh_bien_rows, ky_tu_rows, quality_decision_rows = [], [], []

    for r in rows:
        sid = r["source_image_id"]
        p = plan_by_id[sid]
        anh_bien_rows.append({
            "source": Y2_SOURCE_NAME,
            "source_file": scan_result["ten_tep_nguon"],
            "source_member": r["source_member"],
            "source_image_id": sid,
            "original_split": r["original_split"],
            "parent_id": p["parent_id"],
            "group_id": p["group_id"],
            "width_px": r["width_px"],
            "height_px": r["height_px"],
            "pixel_sha256": r["pixel_sha256"],
            "status": p["status"],
            "reason_codes": p["reason_codes"],
            "yolo2_final_split": p["final_split"],
            "exif_orientation": r["exif_orientation"],
            "label_member": r["label_member"],
            "box_count_total": r["box_count_total"],
            "box_count_valid": r["box_count_valid"],
            "dhash": r["dhash"],
        })
        if p["status"] != "giu" or r["high_iou_pairs"] or r["exif_orientation"] != 1:
            quality_decision_rows.append({
                "level": "plate",
                "source_image_id": sid,
                "source_box_id": "",
                "yolo2_status": p["status"],
                "ocr_can_use_valid_boxes": (not r["image_error"]) and (not r["pixel_label_conflict"]) and r["box_count_valid"] > 0,
                "reason_codes": p["reason_codes"],
                "note": "OCR tu quyet dinh theo chinh sach tung ky tu, khong bi ep loai ca bien neu chi 1 khung loi",
            })
        for b in r["boxes"]:
            box_status_ocr = "giu" if (not r["image_error"] and b["bbox_status"] == "hop_le") else "tam_giu_ngoai"
            box_reasons = list(b["reason_codes"])
            if r["image_error"]:
                box_reasons.append("anh_khong_doc_duoc")
            char_str = b["character"] if (b["bbox_status"] != "khung_loi" or b["character"]) else ""
            ky_tu_rows.append({
                "source": Y2_SOURCE_NAME,
                "source_file": scan_result["ten_tep_nguon"],
                "source_member": r["source_member"],
                "source_image_id": sid,
                "original_split": r["original_split"],
                "parent_id": p["parent_id"],
                "group_id": p["group_id"],
                "width_px": r["width_px"],
                "height_px": r["height_px"],
                "pixel_sha256": r["pixel_sha256"],
                "status": box_status_ocr,
                "reason_codes": ";".join(box_reasons),
                "source_box_id": b["source_box_id"],
                "source_line_index": b["source_line_index"],
                "original_class_id": b["original_class_id"],
                "character": char_str,
                "x1_px": b["x1_px"],
                "y1_px": b["y1_px"],
                "x2_px": b["x2_px"],
                "y2_px": b["y2_px"],
                "bbox_status": b["bbox_status"],
                "raw_yolo_cx": b["raw_yolo_cx"],
                "raw_yolo_cy": b["raw_yolo_cy"],
                "raw_yolo_w": b["raw_yolo_w"],
                "raw_yolo_h": b["raw_yolo_h"],
                "in_ocr_32": char_str in Y2_OCR_32_SET,
            })
            if b["bbox_status"] != "hop_le":
                quality_decision_rows.append({
                    "level": "box",
                    "source_image_id": sid,
                    "source_box_id": b["source_box_id"],
                    "yolo2_status": "da_sua_bien" if b["source_box_id"] in repair_by_box else "tam_giu_ngoai",
                    "ocr_can_use_valid_boxes": b["source_box_id"] in repair_by_box,
                    "reason_codes": ";".join(b["reason_codes"]),
                    "note": "Quyet dinh muc khung; khong tu ep OCR loai cac khung hop le khac tren cung bien",
                })

    y2_write_csv(ocr_dir / "anh_bien_goc.csv", anh_bien_rows, anh_bien_fields)
    y2_write_csv(ocr_dir / "ky_tu_goc.csv", ky_tu_rows, ky_tu_fields)
    y2_write_csv(
        ocr_dir / "quan_he_nhom.csv",
        edges,
        ["edge_id", "source_image_id_a", "source_image_id_b", "split_a", "split_b",
         "evidence_type", "confirmation_status", "label_conflict", "decision_code", "note"],
    )
    y2_write_csv(
        ocr_dir / "quyet_dinh_chat_luong.csv",
        quality_decision_rows,
        ["level", "source_image_id", "source_box_id", "yolo2_status", "ocr_can_use_valid_boxes", "reason_codes", "note"],
    )
    y2_write_json(
        ocr_dir / "thong_tin_phien_ban.json",
        {
            "schema_version": Y2_SCHEMA_VERSION,
            "created_at": y2_now_iso(),
            "export_status": export_scope_status,
            "rule_version": Y2_VERSION,
            "source": Y2_SOURCE_NAME,
            "source_file": scan_result["ten_tep_nguon"],
            "source_metadata_sha256": scan_result["metadata_zip"],
            "decoder_versions": scan_result["thu_vien_giai_ma"],
            "seed": scan_result["seed"],
            "csv_encoding": "utf-8-sig",
            "delimiter": ",",
            "source_line_index_base": 0,
            "bbox_format": "xyxy_pixels_half_open_in_original_image_coordinates",
            "coordinate_system": "raw_stored_pixels_no_exif_rotation",
            "yolo_to_xyxy_formula": "x1=(cx-w/2)*W, y1=(cy-h/2)*H, x2=(cx+w/2)*W, y2=(cy+h/2)*H",
            "unknown_character": "",
            "boolean_values": ["true", "false"],
            "so_anh_bien_goc": len(anh_bien_rows),
            "so_khung_ky_tu_goc": len(ky_tu_rows),
        },
    )
    y2_write_text_atomic(
        ocr_dir / "README.md",
        "# Gói bàn giao metadata gốc từ YOLO2 cho OCR\n\n"
        "- `anh_bien_goc.csv`: toàn bộ ảnh nguồn Roboflow đã xét (kể cả ảnh tạm giữ ngoài ở YOLO2).\n"
        "- `ky_tu_goc.csv`: mỗi dòng là 1 khung ký tự gốc (đánh số dòng từ 0, bảo toàn `original_class_id` và `character`, KHÔNG dùng nhãn lớp `0` của detector).\n"
        "- Tọa độ `x1_px, y1_px, x2_px, y2_px` theo hệ pixel gốc (không tự xoay EXIF), quy ước nửa mở `[x1, x2) x [y1, y2)`; cột `raw_yolo_*` lưu nguyên gốc YOLO chuẩn hóa.\n"
        "- `quan_he_nhom.csv`: OCR chỉ kế thừa cạnh có `confirmation_status=da_xac_nhan`; không tự kế thừa cạnh `dhash_goi_y`.\n"
        "- `quyet_dinh_chat_luong.csv`: tách riêng mức biển (`plate`) và mức khung (`box`) để OCR lọc theo từng ký tự, kể cả trên `Roboflow` `test` theo chính sách riêng của OCR.\n",
    )

    # 3. Kiểm chứng độc lập bộ xuất (Mốc D)
    verify = y2_verify_exported_dataset(dataset_dir, zip_out_path or None)
    # Cập nhật bảng trước–sau thực xuất
    actual_table = []
    for sp in ["train", "val", "test"]:
        ba_plan = next(r for r in plan_result["truoc_sau"] if r["tap"] == sp)
        actual_table.append({
            "tap": sp,
            "anh_ban_dau": ba_plan["anh_ban_dau"],
            "anh_da_xuat": exported_counts[sp],
            "anh_tam_giu_ngoai_hoac_loai": ba_plan["anh_ban_dau"] - exported_counts[sp],
            "khung_ban_dau": ba_plan["khung_hop_le_ban_dau"],
            "khung_da_xuat": exported_box_counts[sp],
            "trang_thai_so_lieu": export_scope_status,
        })
    actual_table.append({
        "tap": "tong",
        "anh_ban_dau": sum(r["anh_ban_dau"] for r in actual_table),
        "anh_da_xuat": sum(r["anh_da_xuat"] for r in actual_table),
        "anh_tam_giu_ngoai_hoac_loai": sum(r["anh_tam_giu_ngoai_hoac_loai"] for r in actual_table),
        "khung_ban_dau": sum(r["khung_ban_dau"] for r in actual_table),
        "khung_da_xuat": sum(r["khung_da_xuat"] for r in actual_table),
        "trang_thai_so_lieu": export_scope_status,
    })
    y2_write_csv(report / "truoc_sau_thuc_xuat.csv", actual_table)

    res = {
        "phien_ban_ma": Y2_VERSION,
        "thoi_diem": y2_now_iso(),
        "trang_thai_xuat": export_scope_status,
        "thu_muc_yolo2": str(dataset_dir),
        "tep_zip_yolo2": zip_out_path,
        "thu_muc_ban_giao_ocr": str(ocr_dir),
        "truoc_sau_thuc_xuat": actual_table,
        "kiem_chung": verify,
        "thoi_gian_giay": round(time.monotonic() - t0, 3),
    }
    y2_write_json(verify_saved, res)
    return res


def y2_verify_exported_dataset(dataset_dir, zip_path=None):
    """Kiểm chứng độc lập toàn bộ nhãn, cặp ảnh–nhãn, nhóm qua tập, ảnh đọc được và nạp sau giải nén ở thư mục khác."""
    dataset_dir = Path(dataset_dir)
    manifest = y2_read_csv(dataset_dir / "manifest.csv")
    cfg = yaml.safe_load((dataset_dir / "data.yaml").read_text(encoding="utf-8"))
    if int(cfg.get("nc", -1)) != 1 or cfg.get("names", {}).get(0) != "char_num":
        raise AssertionError(f"data.yaml không khớp 1 lớp 0: char_num: {cfg}")

    total_imgs_disk, total_lbls_disk, total_boxes_read, decoded_ok = 0, 0, 0, 0
    for sp in ["train", "val", "test"]:
        imgs = sorted(p for p in (dataset_dir / "images" / sp).iterdir() if p.is_file())
        lbls = sorted(p for p in (dataset_dir / "labels" / sp).iterdir() if p.is_file())
        if [p.stem for p in imgs] != [p.stem for p in lbls]:
            raise AssertionError(f"Lệch cặp stem ảnh–nhãn trong tập {sp}.")
        total_imgs_disk += len(imgs)
        total_lbls_disk += len(lbls)
        for lbl_p in lbls:
            for ln in lbl_p.read_text(encoding="utf-8").splitlines():
                if not ln.strip():
                    continue
                parts = ln.split()
                if len(parts) != 5 or parts[0] != "0":
                    raise AssertionError(f"Nhãn xuất sai định dạng hoặc khác lớp 0 tại {lbl_p}: {ln}")
                cx, cy, bw, bh = map(float, parts[1:])
                if not all(math.isfinite(v) for v in (cx, cy, bw, bh)):
                    raise AssertionError(f"Số không hữu hạn tại {lbl_p}")
                if bw <= 0 or bh <= 0 or min(cx - bw / 2, cy - bh / 2) < -1e-5 or max(cx + bw / 2, cy + bh / 2) > 1 + 1e-5:
                    raise AssertionError(f"Khung xuất vượt biên tại {lbl_p}: {ln}")
                total_boxes_read += 1
        for img_p in imgs:
            with Image.open(img_p) as im:
                im.load()
            decoded_ok += 1

    if total_imgs_disk != len(manifest):
        raise AssertionError(f"Số ảnh trên đĩa ({total_imgs_disk}) != manifest ({len(manifest)}).")
    if total_boxes_read != sum(int(r["box_count"]) for r in manifest):
        raise AssertionError("Tổng số khung đọc lại khác manifest.")

    # Kiểm tra 0 nhóm xác nhận qua nhiều tập trên manifest xuất
    grp_splits = defaultdict(set)
    for r in manifest:
        grp_splits[r["group_id"]].add(r["final_split"])
    cross_groups = sum(len(s) > 1 for s in grp_splits.values())
    if cross_groups != 0:
        raise AssertionError(f"Bộ xuất còn {cross_groups} nhóm xác nhận qua nhiều tập!")

    # Thử giải nén / di dời sang thư mục tạm hoàn toàn khác và nạp data.yaml
    reloc_ok = False
    reloc_checked_splits = {}
    with tempfile.TemporaryDirectory(prefix="kiem_tra_nap_yolo2_") as tmp_dir:
        tmp_root = Path(tmp_dir)
        if zip_path and Path(zip_path).is_file():
            with ZipFile(zip_path) as zin:
                for info in zin.infolist():
                    p = PurePosixPath(info.filename)
                    if p.is_absolute() or ".." in p.parts or "\\" in info.filename:
                        raise AssertionError(f"ZIP xuất chứa đường dẫn không an toàn: {info.filename}")
                zin.extractall(tmp_root)
            reloc_ds = tmp_root / "yolo2_char_num"
        else:
            reloc_ds = tmp_root / "yolo2_reloc"
            shutil.copytree(dataset_dir, reloc_ds)
        rcfg = yaml.safe_load((reloc_ds / "data.yaml").read_text(encoding="utf-8"))
        base = (reloc_ds / rcfg["path"]).resolve()
        for sp in ["train", "val", "test"]:
            sp_img_dir = (base / rcfg[sp]).resolve()
            sp_lbl_dir = (base / rcfg[sp].replace("images", "labels", 1)).resolve()
            if not sp_img_dir.is_dir() or not sp_lbl_dir.is_dir():
                raise AssertionError(f"Đường dẫn sau giải nén không tồn tại: {sp_img_dir}")
            reloc_checked_splits[sp] = len(list(sp_img_dir.iterdir()))
        reloc_ok = sum(reloc_checked_splits.values()) == len(manifest)
        if not reloc_ok:
            raise AssertionError("Số ảnh nạp sau giải nén không khớp manifest.")

    return {
        "dat_toan_bo_kiem_chung": True,
        "so_anh_kiem_tra_giai_ma": f"{decoded_ok}/{total_imgs_disk} (100% ảnh xuất)",
        "so_tep_nhan_doc_lai": total_lbls_disk,
        "tong_khung_lop_0_hop_le": total_boxes_read,
        "nhom_xac_nhan_qua_tap": cross_groups,
        "nap_sau_giai_nen_thu_muc_khac": reloc_ok,
        "so_anh_theo_tap_sau_giai_nen": reloc_checked_splits,
    }


## 1. Nguồn Roboflow chứa những gì? (Mốc A)

**Câu hỏi cần trả lời:** Tệp `roboflow_ocr_v1_yolov8.zip` có khớp đúng danh tính dự án `licence-plate-vn/ocr-stypd` phiên bản 1 không? Số lượng ảnh theo `train`/`val`/`test` có khớp mốc kiểm kê `20.262` ảnh không? Và quan trọng nhất: **`data.yaml` khai báo bao nhiêu lớp, từng ID có thực sự là khung ký tự trên ảnh thật hay không?**

> **Lưu ý:** Tuyệt đối không đoán số lớp hay ý nghĩa ID trước khi xem kết quả và ảnh minh họa thật bên dưới. Nếu phát hiện nhãn không phải khung ký tự như dự kiến, cần dừng và báo mâu thuẫn trước khi chuyển đổi sang lớp `0`.


In [ ]:
# @title Mốc A — Kiểm tra danh tính, bảng lớp gốc và vẽ ảnh đối chiếu thật
Y2_MOC_A = y2_step1_inspect_source(
    THU_MUC_DU_AN, NGUON_ROBOFLOW, PHIEN_BAN_XUAT, SO_ANH_THU, mode=CHE_DO, seed=SEED
)

if "thong_bao" in Y2_MOC_A:
    print(Y2_MOC_A["thong_bao"])
else:
    print("Thời điểm:", Y2_MOC_A["thoi_diem"], "|", Y2_MOC_A.get("cach_doc", "Đã kiểm tra từ tệp ZIP nguồn."))
    print("Danh tính nguồn:", Y2_MOC_A["danh_tinh"]["trang_thai"], "— Bằng chứng:", Y2_MOC_A["danh_tinh"]["bang_chung"])
    print("Thư viện giải mã:", Y2_MOC_A["thu_vien"])
    print("Số tệp nhãn mồ côi toàn ZIP:", Y2_MOC_A["so_nhan_mo_coi"], "| Ảnh ngoài cấu trúc:", Y2_MOC_A["so_anh_ngoai_cau_truc"])

    y2_show_markdown("### Đối chiếu số lượng tệp theo split với mốc kiểm kê 20.262")
    y2_display_table(Y2_MOC_A["bang_split"], {
        "tap": "Tập", "so_tep_anh_nguon": "Tệp ảnh nguồn", "so_tep_nhan_khop": "Tệp nhãn khớp",
        "moc_kiem_ke": "Mốc kiểm kê", "chenh_lech_voi_moc": "Chênh lệch",
    })

    y2_show_markdown(f"### Bảng lớp gốc đọc từ `{Y2_MOC_A['yaml_member']}` (`nc = {Y2_MOC_A['nc']}`, ngoài 32 lớp OCR: `{Y2_MOC_A['so_lop_ngoai_ocr32']}`)")
    y2_display_table(Y2_MOC_A["bang_lop_goc"], {
        "original_class_id": "ID gốc", "raw_name": "Tên trong data.yaml",
        "character": "Ký tự giải mã", "in_ocr_32": "Thuộc 32 lớp OCR",
    })

    y2_show_markdown("### Ảnh mẫu thật kèm khung và `ID:ký tự` gốc (Mốc A)")
    y2_show_image(Y2_MOC_A["anh_doi_chieu_moc_a"])

    _tong_row = next(r for r in Y2_MOC_A["bang_split"] if r["tap"] == "tong")
    _nhan_xet_a = (
        f"**Nhận xét Mốc A:** Đã đọc đúng nguồn với **{ _tong_row['so_tep_anh_nguon']:,} tệp ảnh** "
        f"(chênh lệch so với mốc 20.262: `{_tong_row['chenh_lech_voi_moc']:+d}`), "
        f"`data.yaml` có **{Y2_MOC_A['nc']} lớp gốc** ({Y2_MOC_A['so_lop_ngoai_ocr32']} lớp ngoài phạm vi 32 ký tự OCR — vẫn giữ nguyên ở YOLO2). "
        "Hãy kiểm tra trực tiếp lưới ảnh trên; chỉ bật `XAC_NHAN_BANG_LOP_GOC = True` khi xác nhận các khung đúng là từng ký tự biển số."
    )
    y2_show_markdown(_nhan_xet_a)


## 2. Ảnh và nhãn có khớp nhau không? (Mốc B & C)

**Câu hỏi cần trả lời:** Toàn bộ ảnh trong phạm vi xét có giải mã được không? Có nhãn thiếu, nhãn rỗng, khung sai cú pháp/vượt biên, dòng khung lặp, khung chồng lấn bất thường (`IoU >= 0.85`) hoặc thẻ xoay EXIF hay không?

- Tiến độ được lưu từng phần `.jsonl.gz` (ghi tệp tạm rồi đổi tên) và gắn chặt với dấu metadata ZIP + phiên bản thư viện giải mã. Khi chạy lại sau gián đoạn, hệ thống đọc tiếp từ phần còn lại mà không mất hay lặp mẫu; khi nguồn hoặc quy tắc thay đổi, cache cũ bị từ chối.
- Không tự clip khung vượt biên, không tự coi nhãn rỗng là ảnh nền, không tự xóa khung chồng lấn cao.


In [ ]:
# @title Mốc B/C — Quét ảnh–nhãn có lưu tiến độ hoặc mở kết quả đã lưu
Y2_QUET = y2_step2_scan_dataset(
    THU_MUC_DU_AN,
    NGUON_ROBOFLOW,
    PHIEN_BAN_XUAT,
    limit=SO_ANH_THU,
    mode=CHE_DO,
    seed=SEED,
    batch_size=KICH_THUOC_PHAN,
    local_root=THU_MUC_TAM_SSD,
    workers=SO_LUONG,
)

if "thong_bao" in Y2_QUET:
    print(Y2_QUET["thong_bao"])
elif Y2_QUET.get("trang_thai") == "tam_dung_giua_chung":
    print(f"Đang tạm dừng ở {Y2_QUET['da_luu']:,}/{Y2_QUET['muc_tieu']:,} ảnh; chạy lại ô này để tiếp tục.")
else:
    _tag = "TOÀN BỘ NGUỒN" if Y2_QUET["toan_bo"] else f"MẪU THỬ ({Y2_QUET['so_anh_da_xet']:,}/{Y2_QUET['so_anh_nguon']:,} ảnh — CHƯA SUY RỘNG TOÀN NGUỒN)"
    y2_show_markdown(f"**Phạm vi:** {_tag} · **Thời điểm:** `{Y2_QUET['thoi_diem']}`")
    print(Y2_QUET.get("cach_doc", "Đã hoàn tất quét và lưu chi tiết ảnh–nhãn."))
    print(f"Đọc tiếp từ tiến độ cũ: {Y2_QUET['doc_tiep_tu']:,} ảnh | Số luồng: {Y2_QUET['so_luong']} | Thời gian (giây): {Y2_QUET['thoi_gian_giay']}")
    y2_show_markdown("### Ví dụ ngẫu nhiên ảnh–nhãn gốc trong phạm vi vừa xét")
    y2_show_image(Y2_QUET["anh_mau_kiem_tra"])


## 3. Dữ liệu có đặc điểm gì?

**Câu hỏi cần trả lời:** Quy mô ảnh và tổng khung ký tự phân bố ra sao theo từng tập? Kích thước ảnh phổ biến nào chiếm ưu thế? Mỗi ảnh thường có bao nhiêu ký tự, kích thước/tỷ lệ khung ký tự ra sao, và tần suất các ký tự gốc (trước khi chuyển detector về lớp `0`) như thế nào?

Mỗi biểu đồ đều có bảng CSV đi kèm trong thư mục báo cáo, ghi rõ đơn vị, mẫu số và phạm vi.


In [ ]:
# @title Tổng hợp thống kê và hiển thị bảng + biểu đồ đặc điểm dữ liệu
Y2_THONG_KE = y2_step3_stats_and_charts(Y2_QUET, mode=CHE_DO)

if "thong_bao" in Y2_THONG_KE:
    print(Y2_THONG_KE["thong_bao"])
else:
    _scope_lbl = "TOÀN BỘ NGUỒN" if Y2_THONG_KE["toan_bo"] else f"MẪU THỬ ({Y2_THONG_KE['so_anh_da_xet']:,} ảnh)"
    y2_show_markdown(f"### 3.1. Quy mô theo split ({_scope_lbl})")
    y2_display_table(Y2_THONG_KE["quy_mo_theo_split"], {
        "tap": "Tập", "so_tep_anh_nguon": "Ảnh nguồn", "so_anh_da_xet": "Ảnh đã xét",
        "anh_doc_duoc": "Ảnh đọc được", "anh_loi_doc": "Ảnh lỗi",
        "anh_thieu_hoac_loi_tep_nhan": "Thiếu/lỗi tệp nhãn", "anh_nhan_rong": "Nhãn rỗng",
        "anh_co_khung_loi": "Ảnh có khung lỗi", "tong_khung_hop_le": "Tổng khung hợp lệ",
    })
    y2_show_image(Y2_THONG_KE["bieu_do"].get("quy_mo_split", ""))

    y2_show_markdown(f"### 3.2. Kích thước ảnh phổ biến (tổng {Y2_THONG_KE['tong_so_kich_thuoc_khac_nhau']:,} kích thước khác nhau)")
    y2_display_table(Y2_THONG_KE["kich_thuoc_anh_top"][:10], {
        "kich_thuoc": "Kích thước", "so_anh": "Số ảnh", "ty_le_phan_tram": "Tỷ lệ (%)",
    })
    y2_show_image(Y2_THONG_KE["bieu_do"].get("kich_thuoc_anh", ""))

    y2_show_markdown("### 3.3. Phân bố số khung ký tự trên mỗi ảnh")
    y2_display_table(Y2_THONG_KE["phan_bo_so_khung"], {
        "nhom_mo_ta": "Số khung hợp lệ/ảnh", "so_anh": "Số ảnh", "ty_le_phan_tram": "Tỷ lệ (%)",
    }, max_rows=15)
    y2_show_image(Y2_THONG_KE["bieu_do"].get("phan_bo_so_khung", ""))

    y2_show_markdown("### 3.4. Đặc điểm kích thước & tỷ lệ khung ký tự (chỉ mô tả, chưa đặt ngưỡng loại)")
    y2_display_table(Y2_THONG_KE["tom_tat_kich_thuoc_khung"], {
        "chi_so": "Chỉ số", "so_khung": "Số khung", "trung_binh": "TB", "do_lech_chuan": "ĐLC",
        "min": "Min", "p10": "P10", "p50": "Trung vị", "p90": "P90", "max": "Max",
    })
    y2_show_image(Y2_THONG_KE["bieu_do"].get("kich_thuoc_khung", ""))

    y2_show_markdown("### 3.5. Tần suất ký tự nguồn (từ `original_class_id` trước khi gộp detector về lớp `0`)")
    y2_display_table(Y2_THONG_KE["tan_suat_ky_tu_nguon"], {
        "original_class_id": "ID gốc", "character": "Ký tự", "in_ocr_32": "Thuộc 32 lớp OCR",
        "so_khung": "Số khung", "ty_le_phan_tram": "Tỷ lệ (%)",
    })
    y2_show_image(Y2_THONG_KE["bieu_do"].get("tan_suat_ky_tu", ""))

    _top_sz = Y2_THONG_KE["kich_thuoc_anh_top"][0] if Y2_THONG_KE["kich_thuoc_anh_top"] else None
    _top_txt = f"Nhóm kích thước phổ biến nhất là **`{_top_sz['kich_thuoc']}`** ({_top_sz['ty_le_phan_tram']:.2f}% ảnh đọc được)." if _top_sz else ""
    y2_show_markdown(
        f"**Nhận xét Mục 3 ({_scope_lbl}):** Tổng cộng **{Y2_THONG_KE['tong_khung_hop_le']:,} khung ký tự hợp lệ** trên **{Y2_THONG_KE['so_anh_da_xet']:,} ảnh**. "
        f"{_top_txt} Toàn bộ CSV đi kèm được lưu tại `{Y2_THONG_KE['bao_cao']}`."
    )


## 4. Có ảnh liên quan giữa các tập không?

**Câu hỏi cần trả lời:** Có bao nhiêu ảnh bị lỗi/trùng (tính theo **số ảnh duy nhất**)? Có bao nhiêu nhóm trùng pixel, biến thể cùng ảnh gốc `.rf.`, và gợi ý gần giống `dHash` đi qua nhiều tập? Có ca nào trùng pixel nhưng **mâu thuẫn nhãn** không?

- Bảng **lỗi ảnh duy nhất** và bảng **số nhóm trùng** được tách riêng, không cộng lẫn.
- `dHash` chỉ là gợi ý cần xem bảng ảnh đối chiếu, **không dùng tự loại hoặc tự nối nhóm**.
- Ảnh trùng pixel nhưng khác nhãn được **giữ riêng để đối chiếu**, không hợp nhất hay bỏ phiếu nhãn tự động.


In [ ]:
# @title Xem bảng lỗi/trùng lặp và tạo bảng ảnh rà soát Ra_soat/
Y2_RA_SOAT = y2_step4_build_review_sheets(Y2_QUET, mode=CHE_DO, local_root=THU_MUC_TAM_SSD)

if "thong_bao" in Y2_THONG_KE:
    print(Y2_THONG_KE["thong_bao"])
else:
    y2_show_markdown("### 4.1. Bảng lỗi và các vấn đề cần xem (đếm theo số ảnh duy nhất)")
    y2_display_table(Y2_THONG_KE["tom_tat_loi"], {
        "loai_van_de": "Nội dung", "so_anh_duy_nhat": "Số ảnh duy nhất",
        "ty_le_tren_anh_xet_phan_tram": "Tỷ lệ (%)", "ghi_chu": "Ghi chú",
    })

    y2_show_markdown("### 4.2. Bảng nhóm trùng và quan hệ ảnh (tách riêng theo đơn vị nhóm)")
    y2_display_table(Y2_THONG_KE["tom_tat_nhom_trung"], {
        "mo_ta": "Loại quan hệ", "muc_xac_nhan": "Mức xác nhận", "so_nhom": "Số nhóm",
        "so_anh_trong_nhom": "Ảnh trong nhóm", "so_nhom_qua_nhieu_tap": "Nhóm qua nhiều tập",
        "so_nhom_mau_thuan_nhan": "Nhóm mâu thuẫn nhãn",
    })
    y2_show_markdown(
        f"**Tổng số thành phần nhóm ĐÃ XÁC NHẬN đi qua nhiều tập:** `{Y2_THONG_KE['so_nhom_xac_nhan_qua_nhieu_tap']:,}` "
        f"(tổng `{Y2_THONG_KE['so_canh_quan_he']:,}` cạnh quan hệ lưu trong `quan_he_nhom.csv`)."
    )

if "thong_bao" in Y2_RA_SOAT:
    print(Y2_RA_SOAT["thong_bao"])
else:
    y2_show_markdown("### 4.3. Danh sách trang ảnh rà soát đã tạo trong `Ra_soat/`")
    y2_display_table([
        {"loai": k, "so_ca": Y2_RA_SOAT["so_luong_ca"][k], "so_trang": len(v)}
        for k, v in Y2_RA_SOAT["trang_anh"].items()
    ], {"loai": "Nhóm rà soát", "so_ca": "Số ca/ô phát hiện", "so_trang": "Số trang ảnh"})
    print("Tệp ghi quyết định rà soát (nếu cần xác nhận thủ công):", Y2_RA_SOAT["tep_quyet_dinh"])


In [ ]:
# @title Xem trực tiếp từng trang ảnh đối chiếu trong Ra_soat/
LOAI_ANH_RA_SOAT = "Nhan_loi_can_xem"  # @param ["Trung_pixel_mau_thuan_nhan", "Trung_pixel_qua_tap", "Nhom_rf_qua_tap", "Goi_y_dhash_qua_tap", "Nhan_loi_can_xem", "Nhan_rong", "Khung_chong_lap_iou_cao"]
TRANG_RA_SOAT = 1  # @param {type:"integer"}

if "trang_anh" not in Y2_RA_SOAT:
    print(Y2_RA_SOAT.get("thong_bao", "Chưa có dữ liệu rà soát."))
else:
    _pages = Y2_RA_SOAT["trang_anh"].get(LOAI_ANH_RA_SOAT, [])
    if not _pages:
        print(f"Không có ca nào trong nhóm '{LOAI_ANH_RA_SOAT}'.")
    elif not (1 <= TRANG_RA_SOAT <= len(_pages)):
        print(f"Chọn trang từ 1 đến {len(_pages)}.")
    else:
        print(f"{LOAI_ANH_RA_SOAT} — trang {TRANG_RA_SOAT}/{len(_pages)}")
        y2_show_image(_pages[TRANG_RA_SOAT - 1], width=1050)


## 5. Giữ lại và xử lý như thế nào?

**Câu hỏi cần trả lời:** Sau khi áp dụng quy tắc xử lý, mỗi tập giữ lại bao nhiêu ảnh và khung ký tự? Những ảnh nào bị tạm giữ ngoài và vì lý do gì?

**Quy tắc xử lý áp dụng (thống nhất với YOLO1):**
1. **Giữ split gốc** của Roboflow khi ảnh hợp lệ và không xung đột nhóm qua tập.
2. **Nhóm đã xác nhận đi qua nhiều tập (`pixel_sha256`, họ `.rf.` hoặc `dHash` đã xác nhận bằng mắt):** ưu tiên **`test → val → train`**, giữ nguyên tên tập ưu tiên cao nhất và tạm giữ ngoài toàn bộ thành viên của nhóm ở các tập ưu tiên thấp hơn.
3. **Trong cùng tập:**
   - Trùng pixel **khác nhãn**: tạm giữ ngoài để đối chiếu (không tự gộp/bỏ phiếu).
   - Trùng pixel **cùng nhãn**: giữ 1 bản đại diện xác định.
   - Biến thể `.rf.` khác pixel trong `train` (không đi sang `val`/`test`): **giữ lại**; trong `val`/`test` giữ 1 đại diện/nhóm theo cấu hình `CHI_MOT_DAI_DIEN_VAL_TEST`.
4. **Chất lượng ảnh–nhãn:** ảnh không đọc được → `loai`; nhãn thiếu / nhãn rỗng chưa xác nhận nền / khung lỗi chưa có quyết định sửa → `tam_giu_ngoai`. Không lọc bỏ khung YOLO2 vì ký tự ngoài 32 lớp OCR.
5. **Không sửa tệp nguồn.** Kết quả ở bước này được gắn nhãn **`PHƯƠNG ÁN DỰ KIẾN`**, chỉ chuyển sang **`ĐÃ XUẤT`** sau Mục 6.


In [ ]:
# @title Lập phương án xử lý, đối chiếu trước–sau và kiểm chứng không còn nhóm xác nhận qua tập
Y2_PHUONG_AN = y2_step5_make_plan(
    Y2_QUET, mode=CHE_DO, single_rep_val_test=CHI_MOT_DAI_DIEN_VAL_TEST
)

if "thong_bao" in Y2_PHUONG_AN:
    print(Y2_PHUONG_AN["thong_bao"])
else:
    y2_show_markdown("### 5.1. Số lượng trước–sau theo split (PHƯƠNG ÁN DỰ KIẾN)")
    y2_display_table(Y2_PHUONG_AN["truoc_sau"], {
        "tap": "Tập", "anh_ban_dau": "Ảnh ban đầu", "anh_du_kien_giu": "Ảnh dự kiến giữ",
        "anh_tam_giu_ngoai_hoac_loai": "Tạm giữ ngoài / loại",
        "khung_hop_le_ban_dau": "Khung ban đầu", "khung_du_kien_giu": "Khung dự kiến giữ",
        "trang_thai_so_lieu": "Trạng thái",
    })
    y2_show_image(Y2_PHUONG_AN["bieu_do_truoc_sau"], width=820)

    if Y2_PHUONG_AN["ly_do_giu_ngoai"]:
        y2_show_markdown("### 5.2. Lý do tạm giữ ngoài / loại (số ảnh duy nhất theo từng mã)")
        y2_display_table(Y2_PHUONG_AN["ly_do_giu_ngoai"], {
            "ma_ly_do": "Mã lý do", "so_anh_duy_nhat": "Số ảnh duy nhất", "ghi_chu": "Ghi chú",
        })
    else:
        print("Không có ảnh nào bị tạm giữ ngoài trong phạm vi này.")

    y2_show_markdown(
        f"**Kiểm tra rò rỉ nhóm đã xác nhận:** `{Y2_PHUONG_AN['so_nhom_xac_nhan_qua_tap_trong_bo_giu']}` nhóm qua nhiều tập trong bộ giữ lại. "
        f"Số khung được chỉnh biên có xác nhận: `{Y2_PHUONG_AN['so_khung_da_chinh_bien']}`.\n\n"
        f"> **Giới hạn:** {Y2_PHUONG_AN['luu_y_gioi_han']}"
    )


## 6. Ảnh đại diện, xuất dữ liệu bàn giao YOLO2/OCR và kiểm chứng (Mốc D & E)

**Câu hỏi cần trả lời:**
1. Ảnh đại diện theo kích thước phổ biến đã được xuất đúng byte gốc và tách riêng khỏi hình vẽ khung chưa?
2. Bộ huấn luyện `yolo2_char_num/` (lớp `0: char_num`) đã đọc lại toàn bộ nhãn, kiểm tra ảnh, kiểm tra `0` nhóm xác nhận qua tập và **nạp thử sau khi giải nén ở một thư mục khác** chưa?
3. Gói bàn giao `Ban_giao_OCR/` đã đầy đủ 7 tệp và bảo toàn nguyên nhãn ký tự gốc theo schema chung trong `Mau_bang/` chưa?


In [ ]:
# @title 6.1. Xuất hoặc xem ảnh đại diện theo kích thước phổ biến
Y2_ANH_DAI_DIEN = y2_step6_representatives(
    THU_MUC_DU_AN,
    Y2_QUET,
    Y2_PHUONG_AN,
    mode=CHE_DO,
    n_sizes=SO_KICH_THUOC_DAI_DIEN,
    per_size=SO_ANH_MOI_KICH_THUOC,
    rep_scope=PHAM_VI_ANH_DAI_DIEN,
    local_root=THU_MUC_TAM_SSD,
)

if "thong_bao" in Y2_ANH_DAI_DIEN:
    print(Y2_ANH_DAI_DIEN["thong_bao"])
else:
    print(f"Đã lưu {Y2_ANH_DAI_DIEN['so_anh_da_xuat']} ảnh đại diện (tập chọn: {Y2_ANH_DAI_DIEN['pham_vi_chon']}) tại:")
    print(" ", Y2_ANH_DAI_DIEN["thu_muc_anh_nop"])
    y2_display_table(Y2_ANH_DAI_DIEN["danh_sach"], {
        "kich_thuoc": "Kích thước", "original_split": "Tập gốc", "group_id": "Nhóm",
        "source_member": "Đường dẫn trong ZIP", "tep_mau_tuong_doi": "Tệp mẫu",
    })
    y2_show_markdown("**Hình kiểm tra có khung (lưu riêng trong báo cáo; thư mục ảnh nộp giữ nguyên byte gốc):**")
    y2_show_image(Y2_ANH_DAI_DIEN["hinh_ve_khung_bao_cao"])


In [ ]:
# @title 6.2. Xuất bộ YOLO2, gói Ban_giao_OCR và chạy kiểm chứng độc lập (Mốc D & E)
Y2_XUAT = y2_step6_export_and_verify(
    THU_MUC_DU_AN,
    Y2_QUET,
    Y2_PHUONG_AN,
    phien_ban_xuat=PHIEN_BAN_XUAT,
    mode=CHE_DO,
    class_mapping_confirmed=XAC_NHAN_BANG_LOP_GOC,
    do_export=XUAT_BO_DU_LIEU,
    make_zip=NEN_ZIP_XUAT,
    allow_trial_export=CHO_PHEP_XUAT_MAU_THU,
    overwrite=GHI_DE_BAN_XUAT,
    local_root=THU_MUC_TAM_SSD,
)

if "thong_bao" in Y2_XUAT:
    print(Y2_XUAT["thong_bao"])
else:
    y2_show_markdown(f"### Kết quả xuất thực tế — trạng thái: `{Y2_XUAT['trang_thai_xuat']}`")
    y2_display_table(Y2_XUAT["truoc_sau_thuc_xuat"], {
        "tap": "Tập", "anh_ban_dau": "Ảnh ban đầu", "anh_da_xuat": "Ảnh đã xuất",
        "anh_tam_giu_ngoai_hoac_loai": "Tạm giữ ngoài / loại",
        "khung_ban_dau": "Khung ban đầu", "khung_da_xuat": "Khung đã xuất (lớp 0)",
        "trang_thai_so_lieu": "Trạng thái",
    })
    y2_show_markdown("### Kết quả kiểm chứng độc lập bộ xuất (Mốc D)")
    y2_display_table([{"hang_muc": k, "ket_qua": v} for k, v in Y2_XUAT["kiem_chung"].items()], {
        "hang_muc": "Hạng mục kiểm chứng", "ket_qua": "Kết quả thực tế",
    })
    print("Bộ huấn luyện YOLO2:", Y2_XUAT["thu_muc_yolo2"])
    if Y2_XUAT["tep_zip_yolo2"]:
        print("Tệp nén YOLO2:", Y2_XUAT["tep_zip_yolo2"])
    print("Gói bàn giao cho OCR:", Y2_XUAT["thu_muc_ban_giao_ocr"])


In [ ]:
# @title 6.3. Tạo mẫu báo cáo tiến độ bàn giao (theo mục 8 Checklist nghiệm thu)
def y2_print_handover_snippet(moc_a, quet, phuong_an, xuat):
    scope = quet.get("pham_vi", "chua_chay") if isinstance(quet, dict) else "chua_chay"
    n_img = quet.get("so_anh_da_xet", 0) if isinstance(quet, dict) else 0
    n_box = Y2_THONG_KE.get("tong_khung_hop_le", 0) if isinstance(Y2_THONG_KE, dict) else 0
    exported = isinstance(xuat, dict) and "thu_muc_yolo2" in xuat
    if exported:
        tong_x = next(r for r in xuat["truoc_sau_thuc_xuat"] if r["tap"] == "tong")
        xuat_txt = f"Có ({xuat['trang_thai_xuat']}): {tong_x['anh_da_xuat']:,} ảnh, {tong_x['khung_da_xuat']:,} khung tại `{xuat['thu_muc_yolo2']}` và `{xuat['thu_muc_ban_giao_ocr']}`"
    else:
        xuat_txt = "Chưa xuất bộ huấn luyện (mới dừng ở bước phân tích / lập phương án)."
    text = (
        f"> **Phần việc:** YOLO2 — phiên bản `{PHIEN_BAN_XUAT}` (`{Y2_VERSION}`)\n"
        f"> **Đã chạy:** phạm vi `{scope}`; `{n_img:,}` ảnh biển số, `{n_box:,}` khung ký tự hợp lệ.\n"
        f"> **Đã kiểm chứng:** danh tính nguồn `{moc_a.get('danh_tinh', {}).get('trang_thai', 'chua_chay') if isinstance(moc_a, dict) else 'chua_chay'}`; "
        f"bằng chứng tại `{quet.get('bao_cao', 'chua_co') if isinstance(quet, dict) else 'chua_co'}`.\n"
        f"> **Đã xuất:** {xuat_txt}\n"
        f"> **Còn thiếu hoặc chưa rõ:** "
        + ("Không còn vướng kỹ thuật; chờ thành viên OCR đối chiếu `Ban_giao_OCR/`." if exported and quet.get("toan_bo") else "Chưa hoàn tất lượt toàn nguồn hoặc chưa xác nhận bảng lớp / rà soát ảnh để xuất.") + "\n"
        f"> **Bước tiếp theo:** "
        + ("Chuyển `CHE_DO = 'xem_bao_cao'`, chạy lại toàn notebook xác nhận mở được báo cáo mà không quét nguồn." if exported else "Xem bảng ảnh Mốc A & `Ra_soat/`, cập nhật `quyet_dinh_ra_soat.json` nếu cần rồi chạy toàn nguồn (`SO_ANH_THU = 0`).")
    )
    y2_show_markdown(text)

y2_print_handover_snippet(Y2_MOC_A, Y2_QUET, Y2_PHUONG_AN, Y2_XUAT)


## Phụ lục — Tự kiểm tra tự động trên dữ liệu giả lập

> **Phân biệt rõ:** Các kiểm tra bên dưới chạy trên một tệp ZIP giả lập nhỏ trong thư mục tạm để xác minh logic mã (khóa cache khi nguồn đổi, chạy tiếp sau gián đoạn, tương đương 1 luồng/2 luồng, bảo toàn nhãn gốc khi xuất lớp `0`, không còn nhóm xác nhận qua tập, chế độ `xem_bao_cao` không mở ZIP nguồn và nạp `data.yaml` sau giải nén ở thư mục khác). **Đây là kiểm tra giả lập, không thay thế kiểm tra trên dữ liệu Roboflow thật.**


In [ ]:
# @title Chạy bộ tự kiểm tra giả lập (mặc định tắt để chế độ xem chạy nhanh)
CHAY_TU_KIEM_TRA_GIA_LAP = False  # @param {type:"boolean"}


def y2_run_synthetic_self_tests():
    """Chạy 8 kiểm tra bất biến trên dữ liệu giả lập (tạm thời, không chạm dữ liệu thật)."""
    results = []
    with tempfile.TemporaryDirectory(prefix="yolo2_selftest_") as td:
        root = Path(td)
        proj = root / "Do_an_bien_so"
        src_zip = proj / "01_Du_lieu_goc/Roboflow/roboflow_ocr_v1_yolov8.zip"
        src_zip.parent.mkdir(parents=True, exist_ok=True)

        def make_img_bytes(color, size=(120, 40)):
            buf = io.BytesIO()
            Image.new("RGB", size, color).save(buf, format="JPEG", quality=95)
            return buf.getvalue()

        img_dup_px = make_img_bytes((20, 40, 60))
        img_conflict = make_img_bytes((90, 20, 20))
        img_rf_a = make_img_bytes((30, 80, 120))
        img_rf_b = make_img_bytes((35, 85, 125))
        img_clean = make_img_bytes((150, 160, 170), (160, 60))
        img_empty = make_img_bytes((200, 200, 200))
        img_badbox = make_img_bytes((50, 50, 50))

        # Lớp 0='0', 1='A', 2='I' (ngoài 32 lớp OCR — YOLO2 vẫn PHẢI giữ!)
        yaml_txt = "nc: 3\nnames: ['0', 'A', 'I']\n"
        readme_ds = "https://universe.roboflow.com/licence-plate-vn/ocr-stypd/dataset/1\n"
        readme_rf = "OCR - v1 2026-01-01\n"

        with ZipFile(src_zip, "w") as z:
            z.writestr("data.yaml", yaml_txt)
            z.writestr("README.dataset.txt", readme_ds)
            z.writestr("README.roboflow.txt", readme_rf)
            # 1. Clean train image có cả ký tự '0' và ký tự 'I' (ngoài 32 lớp OCR)
            z.writestr("train/images/clean_01.jpg", img_clean)
            z.writestr("train/labels/clean_01.txt", "0 0.25 0.50 0.20 0.60\n2 0.70 0.50 0.20 0.60\n")
            # 2. Trùng pixel qua tập (test ưu tiên hơn train)
            z.writestr("test/images/px_same_test.jpg", img_dup_px)
            z.writestr("test/labels/px_same_test.txt", "1 0.50 0.50 0.30 0.50\n")
            z.writestr("train/images/px_same_train.jpg", img_dup_px)
            z.writestr("train/labels/px_same_train.txt", "1 0.50 0.50 0.30 0.50\n")
            # 3. Trùng pixel trong train nhưng mâu thuẫn nhãn -> phải tạm giữ ngoài cả hai
            z.writestr("train/images/conf_a.jpg", img_conflict)
            z.writestr("train/labels/conf_a.txt", "0 0.50 0.50 0.30 0.50\n")
            z.writestr("train/images/conf_b.jpg", img_conflict)
            z.writestr("train/labels/conf_b.txt", "1 0.50 0.50 0.30 0.50\n")
            # 4. Họ biến thể .rf. qua tập (val ưu tiên hơn train)
            z.writestr("valid/images/plate99.rf.aaaaaaaaaaaaaaaa.jpg", img_rf_a)
            z.writestr("valid/labels/plate99.rf.aaaaaaaaaaaaaaaa.txt", "0 0.50 0.50 0.25 0.50\n")
            z.writestr("train/images/plate99.rf.bbbbbbbbbbbbbbbb.jpg", img_rf_b)
            z.writestr("train/labels/plate99.rf.bbbbbbbbbbbbbbbb.txt", "0 0.50 0.50 0.25 0.50\n")
            # 5. Nhãn rỗng -> không tự coi là ảnh nền
            z.writestr("train/images/empty_01.jpg", img_empty)
            z.writestr("train/labels/empty_01.txt", "")
            # 6. Khung vượt biên -> không tự clip
            z.writestr("train/images/oob_01.jpg", img_badbox)
            z.writestr("train/labels/oob_01.txt", "1 0.99 0.50 0.30 0.50\n")

        # Test 1: Mốc A đọc đúng nguồn và bảng lớp
        ma = y2_step1_inspect_source(proj, src_zip, "Ban_test", limit=0, mode="xu_ly_du_lieu", seed=42)
        assert ma["danh_tinh"]["trang_thai"] == "khop_nguon_va_phien_ban" and ma["nc"] == 3
        results.append({"kiem_tra": "1. Mốc A xác minh danh tính & bảng lớp", "ket_qua": "DAT"})

        # Test 2: Chạy tiếp sau gián đoạn (dừng sau batch 1, rồi chạy tiếp)
        p_stop = y2_step2_scan_dataset(
            proj, src_zip, "Ban_test", limit=0, mode="xu_ly_du_lieu", seed=42,
            batch_size=3, local_root=root / "ssd", workers=1, stop_after_batches=1,
        )
        assert p_stop["trang_thai"] == "tam_dung_giua_chung" and p_stop["da_luu"] == 3
        scan1 = y2_step2_scan_dataset(
            proj, src_zip, "Ban_test", limit=0, mode="xu_ly_du_lieu", seed=42,
            batch_size=3, local_root=root / "ssd", workers=1,
        )
        assert scan1["doc_tiep_tu"] == 3 and scan1["so_anh_da_xet"] == 9
        rows_w1 = y2_load_detail_rows(scan1["bao_cao"])
        results.append({"kiem_tra": "2. Chạy tiếp sau gián đoạn không mất/lặp mẫu", "ket_qua": "DAT"})

        # Test 3: Tương đương I/O (1 luồng vs 2 luồng cho cùng kết quả chính xác)
        scan2 = y2_step2_scan_dataset(
            proj, src_zip, "Ban_test_w2", limit=0, mode="xu_ly_du_lieu", seed=42,
            batch_size=3, local_root=root / "ssd2", workers=2,
        )
        rows_w2 = y2_load_detail_rows(scan2["bao_cao"])
        assert rows_w1 == rows_w2
        results.append({"kiem_tra": "3. Tương đương bản ghi giữa 1 luồng và 2 luồng", "ket_qua": "DAT"})

        # Test 4: Thống kê, rà soát, phương án xử lý ưu tiên test -> val -> train & giữ ngoài mâu thuẫn
        st = y2_step3_stats_and_charts(scan1, mode="xu_ly_du_lieu")
        rv = y2_step4_build_review_sheets(scan1, mode="xu_ly_du_lieu", local_root=root / "ssd")
        pl = y2_step5_make_plan(scan1, mode="xu_ly_du_lieu")
        plan_map = {r["source_member"]: r for r in y2_read_csv(Path(scan1["bao_cao"]) / "phuong_an_tung_anh.csv")}
        assert plan_map["test/images/px_same_test.jpg"]["status"] == "giu"
        assert plan_map["train/images/px_same_train.jpg"]["status"] == "tam_giu_ngoai"
        assert plan_map["valid/images/plate99.rf.aaaaaaaaaaaaaaaa.jpg"]["status"] == "giu"
        assert plan_map["train/images/plate99.rf.bbbbbbbbbbbbbbbb.jpg"]["status"] == "tam_giu_ngoai"
        assert plan_map["train/images/conf_a.jpg"]["status"] == "tam_giu_ngoai"
        assert plan_map["train/images/conf_b.jpg"]["status"] == "tam_giu_ngoai"
        assert plan_map["train/images/empty_01.jpg"]["status"] == "tam_giu_ngoai"
        assert plan_map["train/images/oob_01.jpg"]["status"] == "tam_giu_ngoai"
        assert plan_map["train/images/clean_01.jpg"]["status"] == "giu"
        results.append({"kiem_tra": "4. Quy tắc test->val->train, mâu thuẫn nhãn, nhãn rỗng, vượt biên", "ket_qua": "DAT"})

        # Test 5: Xuất ảnh đại diện + YOLO2 (lớp 0 giữ cả 'I') + Ban_giao_OCR (bảo toàn nhãn gốc) + kiểm chứng nạp sau giải nén
        rep = y2_step6_representatives(proj, scan1, pl, mode="xu_ly_du_lieu", local_root=root / "ssd")
        assert rep["so_anh_da_xuat"] >= 1
        ex = y2_step6_export_and_verify(
            proj, scan1, pl, "Ban_test", mode="xu_ly_du_lieu",
            class_mapping_confirmed=True, do_export=True, make_zip=True, local_root=root / "ssd",
        )
        assert ex["kiem_chung"]["dat_toan_bo_kiem_chung"] is True
        assert ex["kiem_chung"]["nap_sau_giai_nen_thu_muc_khac"] is True
        # Kiểm tra clean_01 xuất 2 khung lớp 0 (giữ cả ký tự 'I' ngoài 32 lớp OCR)
        clean_lbl = list((Path(ex["thu_muc_yolo2"]) / "labels/train").glob("*.txt"))
        assert len(clean_lbl) == 1 and len(clean_lbl[0].read_text().strip().splitlines()) == 2
        # Kiểm tra Ban_giao_OCR giữ nguyên toàn bộ 9 ảnh nguồn và ký tự gốc '0', 'A', 'I'
        ocr_dir = Path(ex["thu_muc_ban_giao_ocr"])
        anh_ocr = y2_read_csv(ocr_dir / "anh_bien_goc.csv")
        kt_ocr = y2_read_csv(ocr_dir / "ky_tu_goc.csv")
        assert len(anh_ocr) == 9
        chars_found = {r["character"] for r in kt_ocr}
        assert {"0", "A", "I"}.issubset(chars_found)
        assert all(r["source_line_index"].isdigit() for r in kt_ocr)
        results.append({"kiem_tra": "5. Xuất lớp 0 giữ đủ khung, bảo toàn nhãn gốc OCR & nạp sau giải nén", "ket_qua": "DAT"})

        # Test 6: Đổi tệp nguồn -> cache cũ PHẢI bị từ chối
        with ZipFile(src_zip, "a") as z:
            z.writestr("train/images/extra_new.jpg", img_clean)
            z.writestr("train/labels/extra_new.txt", "0 0.5 0.5 0.2 0.2\n")
        rejected = False
        try:
            y2_step2_scan_dataset(proj, src_zip, "Ban_test", limit=0, mode="xu_ly_du_lieu", seed=42, batch_size=3, local_root=root / "ssd", workers=1)
        except ValueError:
            rejected = True
        assert rejected, "Cache cũ không bị từ chối khi nguồn thay đổi!"
        results.append({"kiem_tra": "6. Nguồn thay đổi làm vô hiệu cache cũ", "ket_qua": "DAT"})

        # Test 7: Chế độ xem_bao_cao mở được báo cáo mà KHÔNG cần tệp ZIP nguồn
        src_zip.unlink()
        v1 = y2_step1_inspect_source(proj, src_zip, "Ban_test", limit=0, mode="xem_bao_cao", seed=42)
        v2 = y2_step2_scan_dataset(proj, src_zip, "Ban_test", limit=0, mode="xem_bao_cao", seed=42)
        v3 = y2_step3_stats_and_charts(v2, mode="xem_bao_cao")
        v4 = y2_step4_build_review_sheets(v2, mode="xem_bao_cao")
        v5 = y2_step5_make_plan(v2, mode="xem_bao_cao")
        v6 = y2_step6_export_and_verify(proj, v2, v5, "Ban_test", mode="xem_bao_cao")
        assert all("thong_bao" not in x for x in [v1, v2, v3, v4, v5, v6])
        results.append({"kiem_tra": "7. Chế độ xem_bao_cao hoạt động hoàn toàn không mở ZIP nguồn", "ket_qua": "DAT"})

        # Test 8: Chế độ xem_bao_cao khi thiếu báo cáo KHÔNG tự quét nguồn
        missing = y2_step2_scan_dataset(proj, src_zip, "Ban_khong_ton_tai", limit=0, mode="xem_bao_cao", seed=42)
        assert "thong_bao" in missing
        results.append({"kiem_tra": "8. Chế độ xem_bao_cao báo thiếu gọn gàng khi chưa có báo cáo", "ket_qua": "DAT"})

    return results


if CHAY_TU_KIEM_TRA_GIA_LAP:
    print("Đang chạy kiểm tra GIẢ LẬP (không dùng dữ liệu thật)...")
    _st_rows = y2_run_synthetic_self_tests()
    y2_display_table(_st_rows, {"kiem_tra": "Hạng mục tự kiểm tra giả lập", "ket_qua": "Kết quả"})
else:
    print("Tự kiểm tra giả lập đang tắt (CHAY_TU_KIEM_TRA_GIA_LAP = False).")
